# Notebook 05 -- rubric_and_smoke_test

**Asks:** Build the scoring rubric and prove it works on twenty rows before spending money.

One written codebook goes to every model. This notebook writes that codebook to
`rubric.json` -- so there is exactly one copy of it in the whole pipeline -- and then
sends it to each configured model on twenty real responses, printing side by side what
the model actually said and what the parser actually made of it.

Twenty rows costs about two cents and a couple of minutes. The full paid pass costs a
great deal more and runs for hours, so a prompt typo, a placeholder model string or a
parsing bug is worth catching here.

**Reads:** `openends_long.csv` (notebook 00)

**Writes:** `rubric.json`, `smoke_test_results.csv`

## Config

In [1]:
# === CONFIG: the only cell you edit ===
# Every switch in the pipeline lives here. Nothing below this cell needs changing.

RUN_MODE    = "full"        # "smoke" = first 20 rows, costs pennies, proves the
                            #           whole chain works end to end.
                            # "full"  = the real corpus.
HUMAN_GOLD  = False         # False = the pre-registered pipeline. Language models
                            #         do the coding, exactly as prereg 6.7 says.
                            #         This is the default and it runs start to
                            #         finish with no human step.
                            # True  = ALSO run the human-coder arm. That arm is a
                            #         DEVIATION from the registration and needs the
                            #         OSF addendum posted first.
WORKDIR     = "/content/retribution_nlp"   # where every file in the chain lives
RANDOM_SEED = 20260629      # frozen. Changing it re-draws every random split.

# --- PER-NOTEBOOK: the three lines below differ in each notebook ---
NB_ID       = "05"                        # this notebook's number
NB_NAME     = "rubric_and_smoke_test"     # this notebook's job, in two words
NB_ASKS     = "Build the scoring rubric and prove it works on twenty rows before spending money."

## Setup

In [2]:
# === SETUP: folders, the manifest, and the two functions every notebook uses ===
# WHAT THIS CELL DOES: makes the working folder, loads the manifest, and defines
# need() and done(). You never edit this cell.
#
# THE MANIFEST is a small file called _manifest.json that lives beside the data.
# Every notebook records what it produced in there. The next notebook reads it to
# check its inputs exist. That is what lets a notebook tell you "run 03 first"
# instead of dying on a missing file three cells later.

import os, json, hashlib, datetime, sys

os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
MANIFEST = os.path.join(WORKDIR, "_manifest.json")

def _load_manifest():
    """Read the manifest, or start an empty one on the first notebook."""
    if os.path.exists(MANIFEST):
        with open(MANIFEST) as f:
            return json.load(f)
    return {"files": {}, "runs": []}

def _sha256(path, cap=8_000_000):
    """First 8 MB is plenty to catch a file that changed underneath us."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        h.update(f.read(cap))
    return h.hexdigest()[:16]

class MissingInput(Exception):
    """Raised when a required upstream file is not there yet."""
    pass

def need(filename, produced_by=None, optional=False):
    """Return the path to an input file, or explain which notebook makes it.

    This is the function that makes the chain navigable. A missing input stops
    the notebook HERE, with a sentence telling you what to run, rather than
    surfacing as a confusing error further down.
    """
    path = os.path.join(WORKDIR, filename)
    if os.path.exists(path):
        rec = _load_manifest()["files"].get(filename, {})
        stamp = rec.get("written_at", "unknown time")
        print(f"  [ok]   {filename}  (written {stamp})")
        return path
    if optional:
        print(f"  [skip] {filename} is not here. This notebook will carry on "
              f"without it and say so in its output.")
        return None
    where = f" Run notebook {produced_by} first." if produced_by else ""
    raise MissingInput(
        f"\n\n  MISSING INPUT: {filename}\n"
        f"  This notebook cannot run without it.{where}\n"
        f"  Files present in {WORKDIR}:\n    "
        + "\n    ".join(sorted(os.listdir(WORKDIR)) or ["(none)"]) + "\n")

def done(filename, rows=None, note=""):
    """Record a file this notebook produced, so later notebooks can find it."""
    path = os.path.join(WORKDIR, filename)
    if not os.path.exists(path):
        raise FileNotFoundError(f"done() called for {filename} but it is not there")
    m = _load_manifest()
    m["files"][filename] = {
        "written_by": f"{NB_ID}_{NB_NAME}",
        "written_at": datetime.datetime.now().isoformat(timespec="seconds"),
        "rows": rows,
        "sha16": _sha256(path),
        "run_mode": RUN_MODE,
        "note": note,
    }
    m["runs"] = [r for r in m["runs"] if r.get("nb") != NB_ID]
    m["runs"].append({"nb": NB_ID, "name": NB_NAME, "run_mode": RUN_MODE,
                      "human_gold": HUMAN_GOLD,
                      "at": datetime.datetime.now().isoformat(timespec="seconds")})
    with open(MANIFEST, "w") as f:
        json.dump(m, f, indent=1)
    size = os.path.getsize(path)
    print(f"  [wrote] {filename}  ({rows if rows is not None else '?'} rows, "
          f"{size:,} bytes)")

print(f"NOTEBOOK {NB_ID} -- {NB_NAME}")
print(f"  {NB_ASKS}")
print(f"  mode: RUN_MODE={RUN_MODE}, HUMAN_GOLD={HUMAN_GOLD}")
print(f"  folder: {WORKDIR}")
_m = _load_manifest()
if _m["runs"]:
    print("  already run in this folder: "
          + ", ".join(sorted(r["nb"] for r in _m["runs"])))
else:
    print("  this folder is empty: notebook 00 is the place to start.")

NOTEBOOK 05 -- rubric_and_smoke_test
  Build the scoring rubric and prove it works on twenty rows before spending money.
  mode: RUN_MODE=full, HUMAN_GOLD=False
  folder: /content/retribution_nlp
  this folder is empty: notebook 00 is the place to start.


In [3]:
# === SETUP (cont.): the packages this notebook needs, and the versions it got ===
# WHAT THIS CELL DOES: installs the two API client libraries a Colab machine does
# not already ship, imports everything used below, and records the versions this
# session actually resolved. It computes nothing and writes no analysis file.
#
# WHY IT RUNS EVERY TIME: a Colab machine is discarded when the runtime
# disconnects, so the install happens at the top of every session.
#
# WHY THE COMPILED CORE IS NEVER PINNED HERE. numpy and pandas ship in the Colab
# image compiled against one another. A pip install that resolves a different
# numpy leaves the already-built wheels pointing at the wrong C struct layout,
# and the next import dies with
#     ValueError: numpy.dtype size changed, may indicate binary incompatibility
# Nothing in this notebook needs a different numpy, so nothing asks for one.
# Reproducibility is served by RECORDING what the session resolved -- printed
# below, and archived with the outputs -- rather than by forcing a downgrade that
# stops the notebook running at all.
#
# IF YOU EVER SEE THAT ERROR: pip cannot swap a compiled module the live kernel
# has already imported. Runtime -> Restart session, then run from the top.
# WHY THESE TWO *ARE* PINNED, WHEN THE COMPILED CORE ABOVE IS NOT.
# The rule is not "never pin", it is "pin what pinning cannot break". numpy and
# pandas ship compiled against the Colab image, so forcing a version there
# breaks the import. anthropic and openai are pure Python with no such coupling,
# and they are the two libraries whose behaviour IS the measurement: they carry
# the sampling settings prereg 6.7 fixes.
#
# THIS IS NOT HYPOTHETICAL. anthropic 1.0.0 reached PyPI on 20 August 2026 and
# REMOVED temperature, top_p and top_k from the Messages methods outright. The
# smoke test run on 24 August installed it unpinned, got 1.0.0, and every Claude
# call died before it left the machine with
#     TypeError: Messages.create() got an unexpected keyword argument 'temperature'
# reported as claude 0/20 while gpt and open both scored 20/20. Four days between
# a library release and a leg of this study silently ceasing to run.
#
# Pinning below 1.0 restores the typed temperature parameter and with it the
# "all three legs greedy" panel. openai is NOT pinned to a number here because
# both legs that use it currently work and guessing a version would risk
# breaking them; instead the resolved version is printed below and written into
# rubric.json, and the line to paste for the paid run is printed with it. Pin it
# from that record before notebook 06, because 06 resumes across sessions: an
# unpinned client that bumps mid-corpus scores half a leg under one library and
# half under another.
!pip install -q "anthropic<1.0" openai

import pandas as pd
import numpy as np
# json  writes rubric.json and parses the model replies
# re    fence-stripping and the truncation repair inside parse_json_safe
# time  sleep between retries: a fixed pause after a parse failure, exponential
#       backoff after a rate limit
import json, re, time, warnings

# ONE CATEGORY, NOT EVERY WARNING. A bare warnings.filterwarnings('ignore')
# switches off every warning for the rest of the session, including the ones a
# reader is entitled to see: a correlation warning that the input is nearly
# constant, a regression warning that the design matrix is ill conditioned, a
# library announcing that a call is about to change under it. RuntimeWarning is
# the only class this notebook has any reason to quiet -- numpy raises it for the
# mean of an empty selection, which happens here on purpose when a leg parsed
# nothing at all and the accounting below asks for its average anyway.
warnings.filterwarnings('ignore', category=RuntimeWarning)

# anthropic  the Claude leg. Imported by name as well as used, because the retry
#            loop catches anthropic.RateLimitError specifically.
# openai     the second-developer leg AND, through its OpenAI-compatible
#            endpoint, the open-weight leg.
import anthropic
from openai import OpenAI

# Colab Secrets, so an API key is never typed into a cell and never lands in the
# .ipynb that goes to OSF with the prompts and the raw output (prereg 6.7).
# Outside Colab this import fails; the key then falls back to an environment
# variable so the notebook still runs rather than stopping on a missing module.
try:
    from google.colab import userdata
except Exception:
    userdata = None

# The tables printed below are wide (nine facets side by side). Without these two
# settings pandas truncates the middle columns with an ellipsis, and a truncated
# smoke-test table is worse than no smoke-test table.
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)

from importlib import metadata as _md
def _ver(name):
    """Not every package exposes __version__, so fall back to the distribution."""
    try:
        return __import__(name).__version__
    except Exception:
        pass
    try:
        return _md.version(name)
    except Exception as e:
        return f'NOT AVAILABLE -- {type(e).__name__}'

CLIENT_VERSIONS = {p: str(_ver(p)) for p in ('anthropic', 'openai')}

# THE PIN ABOVE IS ONLY REAL IF THE KERNEL ACTUALLY PICKED IT UP.
# pip can install anthropic 0.x while the live kernel goes on holding the 1.x it
# imported earlier in the session. Nothing warns you: the install succeeds, the
# import succeeds, and then every Claude call dies twenty rows later with
# TypeError on the temperature argument, because 1.x is what is really loaded.
# Checking the version of the MODULE THAT IS IMPORTED, not the one pip reports,
# is what closes that gap.
_a_major = str(_ver('anthropic')).split('.')[0]
if _a_major != '0':
    raise RuntimeError(
        f'The kernel is holding anthropic {_ver("anthropic")}, but this notebook '
        'needs 0.x.\n  anthropic 1.0.0 removed the temperature parameter, and '
        'prereg 6.7 fixes\n  temperature at 0 on all three legs, so 1.x cannot '
        'run this measurement.\n  pip has installed the right version; the kernel '
        'has not loaded it.\n  Fix: Runtime -> Restart session, then Run all.')
print('Resolved versions (archive these with the outputs):')
print(f'  python     {sys.version.split()[0]}')
for _pkg in ['numpy', 'pandas', 'anthropic', 'openai']:
    print(f'  {_pkg:10s} ' + str(_ver(_pkg)))
# The exact line to paste into notebook 06 for the paid run. Printed rather than
# hardcoded because it has to be the version this session actually resolved, not
# one written down months ago and no longer what pip returns.
print('\n  To pin these for the paid run, replace the install line in 06 with:')
print(f'    !pip install -q "anthropic=={CLIENT_VERSIONS["anthropic"]}" '
      f'"openai=={CLIENT_VERSIONS["openai"]}"')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 46.8 MB/s eta 0:00:00
Resolved versions (archive these with the outputs):
  python     3.13.15
  numpy      2.1.3
  pandas     2.2.3
  anthropic  0.125.0
  openai     2.54.0

  To pin these for the paid run, replace the install line in 06 with:
    !pip install -q "anthropic==0.125.0" "openai==2.54.0"


## Load

In [4]:
# === LOAD: the corpus the smoke test draws its twenty rows from ===
# WHAT THIS CELL DOES: fetches openends_long.csv through need(), checks that the
# three item types survived the export and that the file holds one row per
# participant per item, then applies the >=3-word floor.
#
# WHY THIS NOTEBOOK TOUCHES THE CORPUS AT ALL. The rubric is written from the
# codebook, not from the data, so the corpus is needed here only for real
# passages to test the prompt on. Invented test sentences would prove nothing:
# they are clean, well punctuated, on topic and about the right length, and every
# failure this notebook hunts for lives in the messy end of the real corpus --
# the one-line answer, the very long answer, the answer with no punctuation.
path_open = need('openends_long.csv', produced_by='00')
df = pd.read_csv(path_open)

# Assert the three item_types survived the export. The ranking item is the
# fragile one: the preregistration calls it Q333 while the Qualtrics export
# column is Priorit_Explain, and it is the only one of the three that is NOT
# split by vignette. It is also the item E3 leans on hardest (design section 3),
# so a silent drop would remove the most direct elicitation of retributive
# reasoning and leave the analysis looking merely underpowered. Prereg 5.15
# requires the three items be scored SEPARATELY and never concatenated;
# item_type is what carries that separation through every method and every model.
assert set(df['item_type'].unique()) == {'ranking', 'backward', 'forward'}, \
    f"Unexpected item_type set: {set(df['item_type'].unique())}"

# ONE ROW PER PARTICIPANT PER ITEM, CHECKED HERE RATHER THAN ASSUMED. Every file
# in this chain is keyed on the pair (response_id, item_type), and every number
# computed from those files -- sample sizes, parse rates, the agreement
# statistics -- is only as true as that key is unique. Notebook 00 checks it on
# the file it writes; a file can be appended to or merged after that, so the
# notebook that READS it checks it again. A duplicated key here would put the
# same passage in the twenty-row draw twice and report twenty rows of coverage
# for nineteen passages.
_dups = int(df.duplicated(['response_id', 'item_type']).sum())
assert _dups == 0, (
    f'{_dups} duplicated (response_id, item_type) rows in openends_long.csv. '
    'The corpus must hold exactly one row per participant per item. Re-run '
    'notebook 00 rather than scoring a file that counts a participant twice.')

# --- The >=3-word validity filter (carried over from Dark Side NB02) ---------
# Below three words there is no reasoning to score: "yes", "idk", "he did it"
# carry no facet structure, and a model asked to score them anyway returns
# confident noise that would land in the agreement statistics as agreement. The
# floor is applied here AND re-checked inside score_one and score_facet_solo, so
# a short text reaching a scorer by another route returns None rather than a
# fabricated score.
#
# NOTE WHAT THE FILTER DOES AND DOES NOT DO. It decides which rows are SENT to a
# model. It never deletes a row from the record: every excluded row is still in
# df, and every parse failure among the rows that were sent is kept as an
# explicit unscored row.
MIN_WORDS = 3
df_valid = df[df['n_words'] >= MIN_WORDS].reset_index(drop=True)

print(f'\nCorpus: {len(df)} rows from {df["response_id"].nunique()} participants, '
      f'one row each per item (checked above).')
print(f'  {len(df_valid)} rows are at least {MIN_WORDS} words and can be scored.')
# WHAT THE PER-ITEM COUNTS ARE FOR. They should be roughly balanced across
# ranking, backward and forward. A count far below the other two means an export
# column did not coalesce upstream and that item is being LOST, not that
# participants skipped it -- and the draw below would then quietly test two item
# prompts instead of three.
print('Rows per item type (the smoke draw takes some of each):')
print(df_valid['item_type'].value_counts().to_string())

  [ok]   openends_long.csv  (written unknown time)

Corpus: 2844 rows from 948 participants, one row each per item (checked above).
  2787 rows are at least 3 words and can be scored.
Rows per item type (the smoke draw takes some of each):
item_type
ranking     944
backward    941
forward     902


## Do

### The codebook

One written codebook, sent to every model unchanged. Prereg 6.7 puts the ensemble
"in place of human raters", so the models are raters and this is their codebook. The
next six cells define it: the facet vocabulary, the system prompt, the nine-facet
annotator instruction, the strict JSON schema, the three item prompts, and the one
function that assembles them into the exact message a model receives.

In [5]:
# === THE CANONICAL FACET VOCABULARY (prereg 5.15; design section 2) ===
# WHAT THIS CELL DOES: names the nine facets, in a fixed order, the four levels of
# the single holistic call, and the numeric scale every facet is scored on.
#
# WHY IT IS HERE. This list is the first thing the rubric is written from and the
# first thing the parser checks a reply against, and it is written into
# rubric.json below so every later notebook reads the same nine names in the same
# order instead of retyping them.
#
# Nine multi-label facets, each scored 0.0-1.0 per response per item_type. They
# are NEVER forced to sum to 1 and never collapsed onto one axis: a response can
# be entirely consequentialist (all six retributive facets 0.0), entirely
# vengeful, or a genuine mixture. Co-occurrence is the object of measurement for
# E4, so averaging it away or subtracting one side from the other -- which the
# Dark Side prosocial-minus-dark roll-up literally did -- destroys the quantity.
#
# The order is fixed and this list is canonical: prompts, per-model columns,
# ensemble means, the agreement table and the long export all iterate FACETS, so
# reordering or renaming an entry here silently renames columns in four files.
FACETS = ['desert', 'proportionality', 'censure', 'moral_balance',
          'revenge', 'suffering', 'deterrence', 'incapacitation', 'rehabilitation']
# The six retributive facets, split into principled/bounded (desert,
# proportionality, censure, moral_balance) and vengeful/unbounded (revenge,
# suffering). E4 asks whether a single writer shows both at once.
RETRIBUTIVE   = ['desert', 'proportionality', 'censure', 'moral_balance', 'revenge', 'suffering']
# Scored on the same pass so the retributive facets are measured against a full
# menu of punishment justifications rather than in a vacuum -- but never summed
# into any retributivism total, and never subtracted from one (design section 2).
CONSEQUENTIAL = ['deterrence', 'incapacitation', 'rehabilitation']
# The single holistic call, deliberately kept OUT of the nine-facet vector. It is
# produced by the same pass and the same rater as the facets, so it is
# descriptive coverage and a target for Krippendorff's alpha, a measure of how
# much independent raters agree on a category label -- never corroboration for
# facet-level co-occurrence, which would be the same read counted twice
# (design section 9, rule 6).
HOLISTIC_LEVELS = ['principled', 'vengeful', 'both', 'neither']

# THE SCALE, AS A PAIR OF CONSTANTS THE CODE ACTUALLY READS. "Score from 0.0 to
# 1.0" is an instruction, and a model can read it as 0-10 or as 1-7 and answer 7
# for everything. That is the one fault no count in this notebook would otherwise
# see: the reply is well formed, all nine facets come back, the parse rate is
# twenty out of twenty, and every number in the file is on the wrong scale. Its
# first symptom would arrive weeks later, as a depressed agreement statistic in
# notebook 07 that reads like raters disagreeing. coerce_facets and
# score_facet_solo both check against these two names.
FACET_MIN, FACET_MAX = 0.0, 1.0

print(f'{len(FACETS)} facets: ' + ', '.join(FACETS))
print(f'each scored {FACET_MIN} to {FACET_MAX}; '
      'holistic levels: ' + ' | '.join(HOLISTIC_LEVELS))

9 facets: desert, proportionality, censure, moral_balance, revenge, suffering, deterrence, incapacitation, rehabilitation
each scored 0.0 to 1.0; holistic levels: principled | vengeful | both | neither


In [6]:
# === THE SYSTEM PROMPT, AND WHY IT SAYS "DO NOT REFUSE" ===
# WHICH DESIGN DECISION THIS SECTION IMPLEMENTS
#   Design section 4: one codebook, three scorers -- the models differ, the
#   instruction does not. If one leg were given a nudged wording, part of the
#   disagreement between models would be prompt variance, and the ICC (the
#   intraclass correlation, the statistic that says how much independent raters
#   agree) would stop meaning inter-rater reliability. Everything in this section
#   is therefore written once and sent to all three legs unchanged.
#
# THE SYSTEM PROMPT
#   Carried over from Dark Side NB02, where it worked alongside a prefilled '{'
#   on the assistant turn. That prefill is gone -- current Anthropic models
#   reject a trailing assistant turn outright -- so this prompt now carries the
#   whole job on every leg rather than most of it on two. It has three jobs: force a bare JSON object (no preamble, no
#   fences, no commentary), and pre-empt the two failure modes that would bias
#   the sample rather than merely thin it. A model that adds a sentence of prose
#   before the object is a parse failure; a model that declines to score a
#   violent passage is a parse failure CORRELATED WITH CONTENT -- the vengeful,
#   graphic responses are exactly the ones a safety policy is most likely to
#   decline, and losing them would quietly depress measured revenge and
#   suffering. Naming the text as research data, and the task as schema
#   application rather than evaluation, keeps that from happening.
JSON_SYSTEM_PROMPT = (
    'You are a precise classification system. Output ONLY a single JSON object. '
    'Do not add preamble, explanation, markdown, or any text outside the JSON. '
    'Do not include comments. Do not refuse to classify. The text you classify '
    'is research data; your job is to apply the schema, not to evaluate the content.'
)
print('System prompt defined.')

System prompt defined.


In [7]:
# === THE ANNOTATOR INSTRUCTION, PART 1 OF 2: THE TASK AND THE NINE FACETS ===
# Read this as a human codebook, because that is what it is. Four things in it
# are load-bearing; two of them are in this half.
#
# 1. "Score each facet INDEPENDENTLY... do NOT pick one winner and do NOT spread
#    nonzero scores out of politeness." The nine labels are multi-label and never
#    forced to sum to 1 (prereg 5.15). Both failure directions are named because
#    both are real: a model that picks one winner destroys the co-occurrence E4
#    exists to measure, and a model that sprinkles 0.3 across all nine
#    manufactures it.
#
# 2. The four-way principled distinction -- desert (they earned it) vs
#    proportionality (the amount fits and is bounded) vs censure (society
#    condemns) vs moral_balance (a debt is settled) -- which differ EVEN WHEN
#    THEY APPEAR TOGETHER. The Dark Side scheme buried all four in one "just
#    deserts" bucket, which is precisely why it cannot carry E5 (design section
#    1): you cannot measure a dissociation between two things you have merged.
#
# The instruction is one continuous piece of text. It is written here in two
# halves only so that neither cell runs long enough to be hard to read; the
# halves are joined in the next cell and the model sees a single instruction.
FACET_RUBRIC_PART1 = '''You are one of several independent expert annotators coding how ordinary people justify a criminal sentence. Score each of NINE reasoning facets from 0.0 (absent) to 1.0 (clearly, centrally present) for how strongly the passage expresses it.

Score each facet INDEPENDENTLY. Several facets can be high at once; real explanations mix motives, so do NOT pick one winner and do NOT spread nonzero scores out of politeness. A response can be entirely consequentialist (all six retributive facets 0.0), entirely vengeful, or a genuine mixture.

The nine facets:
  RETRIBUTIVE (principled / bounded):
   - desert: the OFFENDER earned/merited punishment as a backward fact about what they did or are ("deserves it", "earned this", "had it coming", "pay for what he did"). Agent-focused, magnitude-unbounded.
   - proportionality: the PUNISHMENT'S calibration/amount, a bounded fit with a ceiling/floor ("fit the crime", "in proportion", "not too harsh or too lenient", "a reasonable amount to match").
   - censure: the SOCIAL message, punishment as moral condemnation/accountability, satisfiable symbolically ("send a message it's wrong", "society condemns", "reflect the seriousness").
   - moral_balance: restoring a LEDGER, righting the wrong, paying a debt, closure ("pay his debt to society", "balance the scales", "make it right"). About closure, not calibration.
  RETRIBUTIVE (vengeful / unbounded):
   - revenge: retaliation, getting even, symmetric hitting-back ("payback", "eye for an eye", "serves them right", "his turn").
   - suffering: wanting pain beyond any instrumental aim ("should suffer", "rot", "make it miserable", "feel what the victim felt").
  CONSEQUENTIALIST (score the same way; these give the retributive facets a full menu):
   - deterrence: discourage future crime by this offender or others.
   - incapacitation: keep society safe by removing the offender.
   - rehabilitation: change, treatment, reform, a second chance.'''
print(f'Annotator instruction, part 1: {len(FACET_RUBRIC_PART1):,} characters.')

Annotator instruction, part 1: 1,949 characters.


In [8]:
# === THE ANNOTATOR INSTRUCTION, PART 2 OF 2: DISTINCTIONS, STANCE, HOLDOUTS ===
# The other two load-bearing things:
#
# 3. Negation and stance, taught with cases rather than asserted. "I do NOT want
#    him to suffer" is suffering = 0.0; a writer who calls deserving-punishment
#    "just revenge" IN ORDER TO REJECT IT scores desert LOW and revenge LOW; "he
#    deserves NO mercy" is MAX desert, not a negation of it. Token-spotting gets
#    all three backwards, and the third is the one a naive negation rule breaks.
#
# 4. "Do NOT infer a facet from the crime; score only what the TEXT expresses."
#    Every participant read a vignette about the same offence. A model allowed to
#    reason from the offence would score the vignette, not the writer, and the
#    facet scores would converge on a constant with vignette-level noise.
#
# WHAT IS DELIBERATELY ABSENT: the numeric anchor targets. The codebook was
# authored against real corpus quotes with their target scores (design section
# 4), but the SCORED prompt carries only boundary definitions and teaching cases.
# Showing the model "desert 0.9, moral_balance 0.4" next to a near-identical
# corpus quote trains it to reproduce the codebook author's intuitions on easy
# clean cases and inflates agreement exactly where agreement is cheap.
#
# THE OTHER HALF OF THAT RULE, AND WHO ENFORCES IT. Anti-circularity rule 4
# (design section 9, referring to section 8) requires the anchors that informed
# this wording to come from the SEED half of the frozen split, so that the TEST
# half every gate number is computed on cannot leak into the instruction that
# scores it. This notebook holds no anchor list and cannot tell which half an
# anchor came from: notebook 04 is where that is enforced, and it drops any
# codebook anchor the frozen split put in the TEST half and prints the drop.
# What IS checkable here is checked two cells below -- that no corpus passage
# appears verbatim anywhere in the text a model is shown.
FACET_RUBRIC_PART2 = '''Distinguish DESERT (they earned it) from PROPORTIONALITY (the amount fits/is bounded) from CENSURE (condemn / send a message) from MORAL-BALANCE (pay a debt / settle the ledger). These differ even when they appear together.

Handle negation and stance. "I do NOT want him to suffer" is suffering = 0.0. A passage that calls deserving-punishment "just revenge" in order to REJECT it is critiquing desert, not endorsing it (desert LOW, revenge LOW). "He deserves NO mercy" is a MAX-desert statement, not a negation of desert.

Do NOT infer a facet from the crime; score only what the TEXT expresses.

Also make ONE holistic call for the passage as a whole: "principled" | "vengeful" | "both" | "neither".'''

# The two halves rejoined, with the blank line that separates them. From here on
# there is one name, FACET_RUBRIC, and it is the only instruction any leg is ever
# given.
FACET_RUBRIC = FACET_RUBRIC_PART1 + '\n\n' + FACET_RUBRIC_PART2
print(f'Annotator instruction assembled: {len(FACET_RUBRIC):,} characters.')

Annotator instruction assembled: 2,653 characters.


In [9]:
# === THE STRICT JSON SCHEMA ===
# Exactly nine facet keys, one holistic key, one evidence object. Sending the
# literal target shape -- rather than describing it in prose -- is what makes the
# reply parseable often enough that unscored rows stay rare.
#
# WHY `evidence` IS OPTIONAL, AND WHY THAT IS A MEASUREMENT DECISION
#   A mandatory verbatim span turns the scorer into a lexical matcher: it must
#   find a quotable string before it can score a facet nonzero. That suppresses
#   exactly the non-literal vengeful and censure sentiment the ensemble exists to
#   catch where the word list already fails -- "he should never see daylight
#   again" is suffering with no suffering token in it. Making the field optional,
#   and saying so in the user message, means the span requirement never quietly
#   gates revenge and suffering recall (design section 4).
SCHEMA_HINT = (
    '{"desert":0.0,"proportionality":0.0,"censure":0.0,"moral_balance":0.0,'
    '"revenge":0.0,"suffering":0.0,"deterrence":0.0,"incapacitation":0.0,'
    '"rehabilitation":0.0,"holistic":"principled|vengeful|both|neither",'
    '"evidence":{}}'
)
print('Schema defined (nine facet keys, one holistic key, optional evidence).')

Schema defined (nine facet keys, one holistic key, optional evidence).


In [10]:
# === THE THREE ITEM PROMPTS AND THE SINGLE PROMPT-ASSEMBLY PATH ===
# Prereg 5.15: the three open-ends are scored SEPARATELY, never concatenated,
# because the backward pair (why this sentence / why this ranking) and the
# forward item (what should it accomplish) elicit different registers, and that
# difference is itself a measure (E6). Passing the writer's actual prompt tells
# the model what question the passage answers; the user message immediately
# labels it context-only, so "the prompt asked about accomplishment" cannot on
# its own push deterrence up.
#   backward  Q1   = Sentence_OpenEnd1  (split by vignette upstream, coalesced by 00)
#   forward   Q2   = Sentence_OpenEnd2  (split by vignette upstream, coalesced by 00)
#   ranking   Q333 = Priorit_Explain    (single column, new in this study)
ITEM_PROMPTS = {
    'backward': 'Please explain why you recommended this sentence for Darryl.',
    'forward':  'What do you hope this sentence will accomplish?',
    'ranking':  'Please explain why you ranked the reasons for punishment in that order.',
}

# THE PASSAGE WINDOW, AND WHY IT IS A NAME RATHER THAN A NUMBER TYPED TWICE.
# Carried over from Dark Side NB02. The longest item (ranking, Q333) has a median
# of 32 words and a mean of about 44, so 1500 characters clears essentially every
# response in the corpus; the cap exists to stop one pathological entry from
# blowing the context or the bill.
#
# Both prompt builders below read this constant, and it is written into
# rubric.json, which notebook 06 reads before the paid pass. A literal 1500 typed
# into the prompt instead would let this notebook prove one window while the paid
# run used another -- which is the exact divergence that recording the setting
# was supposed to prevent. It is also part of the fingerprint below, because a
# codebook read through a 400-character keyhole is a different instrument from
# the same codebook read whole.
TEXT_TRUNCATION_CHARS = 1500

# Every joint-call score in the pipeline comes through this one function, for
# every model, so there is exactly one place the instruction can be changed and
# no way for two legs to drift apart. Note what the signature does NOT take: a
# model argument. There is nowhere in the prompt path for a leg-specific wording
# to enter, which is the structural form of the "one codebook, three scorers"
# rule rather than a promise about it.
def build_user_prompt(text, item_type):
    # .get with a '' default rather than [] indexing: an unexpected item_type
    # degrades to "no prompt context" instead of raising inside a worker thread,
    # where the exception would be swallowed into an unscored row.
    prompt_text = ITEM_PROMPTS.get(item_type, '')
    # The window is applied identically in build_solo_prompt, so the joint and
    # the split calls read the same passage -- otherwise the halo diagnostic
    # would be comparing scores taken from two different texts.
    #
    # Triple quotes fence the passage so the model can tell instruction from
    # data; chr(34) * 3 builds them without nesting quote characters inside this
    # already quote-heavy piece of string building.
    return (
        FACET_RUBRIC
        + f'\n\nThe writer was answering this prompt (context only; do NOT let it bias the labels): '
          f'"{prompt_text}"'
        + '\n\nPassage to score:\n' + (chr(34) * 3) + '\n'
        + text[:TEXT_TRUNCATION_CHARS] + '\n' + (chr(34) * 3) + '\n\n'
        + 'Return ONLY a JSON object with EXACTLY these keys and 0.0-1.0 facet values '
          '(evidence is OPTIONAL; use <=12-word verbatim spans, and it is fine to omit a '
          'facet from evidence when it is present but not attributable to a single span):\n'
        + SCHEMA_HINT
    )

print(f'Item prompts and build_user_prompt defined (passage window '
      f'{TEXT_TRUNCATION_CHARS} characters; numeric anchors held out; '
      f'evidence optional).')

Item prompts and build_user_prompt defined (passage window 1500 characters; numeric anchors held out; evidence optional).


### The E4 pair, elicited on its own

Desert and revenge are also scored one facet per call, because E4's hypothesis *is*
their association and a single joint read would otherwise produce both sides of it.
The prompt for that pass is defined here so the smoke test can prove it too.

In [11]:
# === WHY THE E4 PAIR IS ELICITED IN SEPARATE, INDEPENDENT CALLS ===
# WHAT THIS CELL AND THE NEXT DO
#   Build a single-facet prompt, used only for desert and revenge, plus a variant
#   with the co-occurrence sentence removed.
#
# THE CIRCULARITY IT CLOSES
#   E4's hypothesis IS the desert x revenge association: does principled desert
#   language appear alongside vengeful language in the same person's writing? In
#   the joint call both numbers come out of one read of one passage, sharing the
#   rater's holistic impression, the same passage window, and a rubric
#   that explicitly instructs "real explanations mix motives". A within-rater
#   halo under a co-occurrence instruction would produce a positive desert x
#   revenge correlation whether or not the writing contains one -- and that is
#   double-dipping, since a single joint read would then be producing both
#   variables whose association is the hypothesis.
#
#   Scoring each facet in its OWN call, with no other-facet context and no
#   co-occurrence instruction, removes the shared read. The joint scores are kept
#   too, and the difference between the two correlations is the halo diagnostic:
#   a correlation that collapses under split scoring was a rating artifact rather
#   than a property of the text.
#
#   This is a registered deviation. Prereg 6.7 says "each model scores every
#   response on the facets" -- one pass, all facets -- so the split-call pass is
#   an ADDITION carried on the OSF addendum and listed in the deviation table in
#   MEASUREMENT_DESIGN.md section 0. The preregistered joint-call estimate is
#   reported in full alongside it; the split-call estimate carries the primary
#   weight.
#
# THE THIRD PASS: NO MIXED ANCHOR
#   The mixed-motive teaching cases in the codebook are the ones that teach
#   co-occurrence. A co-occurrence result that depends on the rubric's own
#   co-occurrence instruction is circular, so mixed_anchor=False strips that
#   sentence and the same passages are scored again. The gap between the two
#   split estimates is the anchor-sensitivity measurement (design section 4).

# The two single-facet definitions, worded to match the joint rubric exactly so
# the only difference between a joint and a split score is the ISOLATION, not the
# construct. Note what is absent: neither definition mentions the other facet, so
# nothing in a desert call can prime a revenge reading or the reverse.
_SOLO_DEFN = {
    'desert':  ('desert', 'the OFFENDER earned/merited punishment as a backward fact about '
                'what they did or are (agent-focused, magnitude-unbounded): "deserves it", '
                '"earned this", "had it coming", "pay for what he did".'),
    'revenge': ('revenge', 'retaliation, getting even, symmetric hitting-back: "payback", '
                '"eye for an eye", "serves them right", "his turn".'),
}
print('Solo facet definitions loaded for: ' + ', '.join(_SOLO_DEFN))

Solo facet definitions loaded for: desert, revenge


In [12]:
# === THE SINGLE-FACET PROMPT ===
# Deliberately parallel to build_user_prompt -- same item-prompt context line,
# same context-only warning, same triple-quoted fence, same TEXT_TRUNCATION_CHARS
# window -- so joint and split scores are read off the same passage under the
# same conditions and the halo delta isolates the one thing that changed.

# The negation/stance teaching case is kept in BOTH variants. It is about reading
# the writer's stance correctly, not about facets co-occurring, so dropping it
# would degrade accuracy rather than test anchor sensitivity.
_SOLO_TEACH = ('Handle negation and stance: a passage that REJECTS or critiques this idea scores '
               'LOW; "he deserves NO mercy" is MAX-desert, not a negation. Do not infer the facet '
               'from the crime; score only what the TEXT expresses.')

# This sentence -- and only this sentence -- is what mixed_anchor toggles. It is
# the co-occurrence instruction: keep it (True) and the split call still tells
# the model that motives mix; remove it (False) and the model is given no reason
# at all to expect the other facet. Scoring both ways quantifies how much of any
# desert x revenge co-occurrence rides on the instruction rather than on the
# writing.
_SOLO_COOCC = (' Real explanations can mix motives, so score this facet on its own terms '
               'regardless of whatever else the passage expresses.')

def build_solo_prompt(text, item_type, facet, mixed_anchor=True):
    name, defn = _SOLO_DEFN[facet]
    teach = _SOLO_TEACH
    coocc = '' if not mixed_anchor else _SOLO_COOCC
    prompt_text = ITEM_PROMPTS.get(item_type, '')
    return (
        f'You are an expert annotator. Score ONLY the single facet "{name}" from 0.0 (absent) '
        f'to 1.0 (clearly, centrally present) for how strongly the passage expresses it.\n\n'
        f'Definition of {name}: {defn}\n{teach}{coocc}\n\n'
        f'Prompt the writer answered (context only; do NOT let it bias the label): "{prompt_text}"\n\n'
        f'Passage:\n' + (chr(34) * 3) + f'\n{text[:TEXT_TRUNCATION_CHARS]}\n' + (chr(34) * 3) + '\n\n'
        f'Return ONLY this JSON object: {{"{name}": 0.0}}'
    )

print('build_solo_prompt defined (desert/revenge; the co-occurrence sentence is '
      'the only thing mixed_anchor moves).')

build_solo_prompt defined (desert/revenge; the co-occurrence sentence is the only thing mixed_anchor moves).


In [13]:
# === SAMPLING SETTINGS: PART OF THE CODEBOOK, NOT OF THE PLUMBING ===
# These two settings sit here, beside the rubric, because they change what a
# model does with the instruction and therefore change the measurement. They are
# written into rubric.json below, and notebook 06 reads them from there before
# the paid pass.
#
# temperature 0.0  Scoring is measurement, not generation. Above zero the same
#                  passage returns a different number on a re-run, which turns
#                  sampling noise into apparent inter-model disagreement and
#                  makes the agreement statistic unrepeatable. Prereg 6.7's
#                  "fixed setting" is this. (Greedy decoding is still not
#                  bit-for-bit deterministic across provider infrastructure,
#                  which is exactly why all raw output is saved rather than
#                  assumed regenerable.)
# max_tokens 900   Headroom for the full nine-facet object plus the holistic call
#                  plus optional evidence spans. Set it too low and long replies
#                  truncate mid-object; parse_json_safe repairs some of that, but
#                  a repaired object can be missing trailing facets, which land
#                  as NaN and read downstream as "facet absent" rather than
#                  "unmeasured".
#
# ALL THREE ADAPTERS READ THESE TWO NAMES. Nothing below carries its own copy of
# the numbers, so editing this line changes what is actually sent -- and the
# rubric records what was actually used rather than what was meant.
GEN_KW = {'temperature': 0.0, 'max_tokens': 900}

# A THIRD FIXED SETTING, AND IT BELONGS BESIDE THE OTHER TWO.
# The OpenAI leg is a GPT-5.x model, and that family accepts temperature ONLY
# when reasoning effort is 'none'; at any real effort it rejects the parameter
# outright. So this line is not a performance knob -- it is what keeps leg 2
# greedy and keeps prereg 6.7's fixed-setting clause true of all three legs at
# once. It is written into rubric.json with temperature and max_tokens, and
# notebook 06 reads it back from there, for the same reason those two are: a
# setting the record does not carry is a setting nobody can check.
GPT_REASONING_EFFORT = 'none'

# HAZARD -- two unrelated things in this pipeline are called SEED.
#   This constant is the DECODING seed handed to OpenAI-compatible endpoints
#   (best-effort determinism on top of temperature 0; Anthropic ignores it).
#   The string 'SEED' in the frozen split is the half of the corpus that supplies
#   embedding reference passages and lexicon expansions, and that is barred from
#   every gate number (design section 8).
#   Changing this integer re-rolls decoding ties. It has nothing to do with the
#   frozen split, and the frozen split must never be re-rolled at all.
SEED = 12345

print(f'Sampling: temperature {GEN_KW["temperature"]}, max_tokens '
      f'{GEN_KW["max_tokens"]}, decoding seed {SEED}, '
      f'gpt reasoning_effort {GPT_REASONING_EFFORT}.')
if GPT_REASONING_EFFORT != 'none':
    print('  WARNING: the OpenAI leg will REJECT temperature at any reasoning')
    print('  effort other than "none", so this run would not be greedy on all')
    print('  three legs and prereg 6.7\'s fixed setting would no longer hold.')
# Said here as well as in notebook 06, because this is where the setting is
# chosen and this is the notebook that proves it before any money is spent.
if GEN_KW['temperature'] != 0.0:
    print('  WARNING: temperature is not 0. Prereg 6.7 fixes the setting, and at')
    print('  any higher temperature the same passage returns a different number on')
    print('  a re-run, which turns sampling noise into apparent disagreement')
    print('  between the raters and makes the agreement statistic unrepeatable.')

Sampling: temperature 0.0, max_tokens 900, decoding seed 12345, gpt reasoning_effort none.


### One rubric, one fingerprint

Everything above is now frozen into a single dictionary and given a short fingerprint.
The fingerprint is what lets a later notebook check that the codebook it is scoring
under is the codebook that was proven here: if a word of the instruction ever changes,
every score produced under the old wording is a different measurement, and the
fingerprint makes that visible instead of invisible.

In [14]:
# === ASSEMBLE THE RUBRIC AND FINGERPRINT IT ===
# WHAT THIS CELL DOES: collects every piece of the codebook into one dictionary
# and takes a short hash over everything in it that reaches a model.
#
# WHY A FINGERPRINT. "One codebook, three scorers" is easy to state and easy to
# break: someone fixes a typo in the annotator instruction three months from now,
# re-runs one leg, and the ensemble is quietly two rubrics wearing one name. The
# fingerprint travels with the scores. Notebook 06 stamps it on every record it
# archives and refuses to re-use an archived call whose fingerprint differs from
# the one in the rubric.json it just loaded, so an edited codebook re-scores
# instead of blending two wordings inside one leg. (Notebook 07, which merges the
# legs and computes the agreement statistic, does not compare fingerprints, so
# two legs scored months apart under different wordings would have to be caught
# by eye in the archive.)
#
# WHAT THE HASH COVERS: exactly what a model is shown, and nothing else. System
# prompt, annotator instruction, schema, the three item prompts, all four pieces
# of the split-call prompt, and the passage window. The split-call pieces are in
# it because the split estimate carries the primary weight for E4, so a changed
# co-occurrence sentence must invalidate a stored score exactly as a changed
# joint instruction does. The window is in it because a codebook read through a
# 400-character keyhole is a different instrument from the same codebook read
# whole. Bookkeeping that never reaches a model -- the run timestamp, the model
# roster -- stays out, so it can change without invalidating comparable scores.
# === DID THE CELLS ABOVE ACTUALLY RUN IN THIS KERNEL? ===
# A restarted Colab session keeps the notebook and throws away every variable in
# it. Run on from the middle after that and Python reports the first missing
# name, which tells you nothing about why it is missing. This turns that into a
# sentence.
#
# THE COMMON CAUSE IS A PINNED INSTALL ON A WARM SESSION. Re-running the install
# cell after the kernel has already imported a package makes pip DOWNGRADE
# something in use, and Colab offers RESTART SESSION. Accepting is correct.
# Running on from where you were is what breaks.
_missing = [_n for _n in ('JSON_SYSTEM_PROMPT', 'FACET_RUBRIC', 'SCHEMA_HINT', 'ITEM_PROMPTS',
           '_SOLO_DEFN', '_SOLO_TEACH', '_SOLO_COOCC',
           'TEXT_TRUNCATION_CHARS', 'GEN_KW', 'GPT_REASONING_EFFORT') if _n not in globals()]
if _missing:
    raise NameError(
        'The cells above have not run in this kernel, so the codebook this cell fingerprints is missing: '
        + ', '.join(_missing)
        + '.\n  Nothing is wrong with the notebook. This is what a restarted Colab'
          '\n  session looks like.'
          '\n  Fix: Runtime -> Run all. Always run this notebook from the top.')

_fingerprint_text = (
    JSON_SYSTEM_PROMPT + FACET_RUBRIC + SCHEMA_HINT
    + json.dumps(ITEM_PROMPTS, sort_keys=True)
    + json.dumps({k: list(v) for k, v in _SOLO_DEFN.items()}, sort_keys=True)
    + _SOLO_TEACH + _SOLO_COOCC
    + f'|passage_window={TEXT_TRUNCATION_CHARS}'
)
RUBRIC_SHA16 = hashlib.sha256(_fingerprint_text.encode('utf-8')).hexdigest()[:16]

RUBRIC = {
    'rubric_fingerprint': RUBRIC_SHA16,
    'written_by':         f'{NB_ID}_{NB_NAME}',
    'facets':             FACETS,
    'retributive':        RETRIBUTIVE,
    'consequential':      CONSEQUENTIAL,
    'holistic_levels':    HOLISTIC_LEVELS,
    'system_prompt':      JSON_SYSTEM_PROMPT,
    'facet_rubric':       FACET_RUBRIC,
    'schema_hint':        SCHEMA_HINT,
    'item_prompts':       ITEM_PROMPTS,
    # The split-call pieces, stored separately so the paid run rebuilds the same
    # solo prompt rather than retyping a near-copy of it.
    'solo_definitions':   {k: list(v) for k, v in _SOLO_DEFN.items()},
    'solo_teaching_case': _SOLO_TEACH,
    'solo_cooccurrence_sentence': _SOLO_COOCC,
    # Settings that change what a model sees or how it answers, and therefore
    # change the measurement. Every one of them is read from the name that this
    # notebook itself used, never retyped, so the file cannot record a setting
    # the smoke test did not actually run under.
    'text_truncation_chars': TEXT_TRUNCATION_CHARS,
    'min_words':             MIN_WORDS,
    'temperature':           GEN_KW['temperature'],
    'max_tokens':            GEN_KW['max_tokens'],
    'gpt_reasoning_effort':  GPT_REASONING_EFFORT,
    'facet_value_range':     [FACET_MIN, FACET_MAX],
    # Stated on the record, because all three are deliberate and all three would
    # look like oversights to a reader who did not know.
    'numeric_anchor_targets_included': False,
    'evidence_span_required':          False,
    'anchor_provenance': ('codebook anchors draw from the SEED half of the frozen '
                          'split (design section 8); enforced in notebook 04, '
                          'which drops any anchor the split put in TEST'),
}
print(f'Rubric assembled. Fingerprint: {RUBRIC_SHA16}')
print(f'  annotator instruction {len(FACET_RUBRIC):,} chars, '
      f'{len(FACETS)} facets, {len(ITEM_PROMPTS)} item prompts, '
      f'passage window {TEXT_TRUNCATION_CHARS}.')

Rubric assembled. Fingerprint: fef0381e46892391
  annotator instruction 2,653 chars, 9 facets, 3 item prompts, passage window 1500.


In [15]:
# === CHECK: NO SCORED PASSAGE APPEARS IN THE INSTRUCTION THAT SCORES IT ===
# WHY THIS IS HERE. Anti-circularity rule 4 (design section 9) holds the rubric's
# anchors to the SEED half of the frozen split, so that no passage a gate number
# is computed on has already been shown to the rater as an example of what a
# facet looks like. A model handed a passage, told what it scores, and then asked
# to score that same passage is not an independent rater of it.
#
# The half of that rule this notebook can check, it checks: whether any corpus
# response is reproduced verbatim inside the text a model is shown. That covers
# every route a passage could take into the prompt -- an anchor pasted in while
# the codebook was being edited, a teaching case grown out of a real answer --
# and it needs nothing beyond the corpus already loaded above. WHICH HALF an
# anchor came from is checked in notebook 04, which drops any codebook anchor the
# frozen split put in the TEST half.
#
# TWELVE WORDS IN A ROW is the floor for a match, and it is a RUN of words rather
# than the whole answer because a codebook anchor is an abridged quote: a leak
# arrives as a fragment of a real response, not as the response entire. Short
# overlaps prove nothing -- the codebook quotes ordinary phrases such as "eye for
# an eye" and "deserves it", and a participant writing one of those is language
# held in common. Twelve consecutive words in common is not coincidence.
MATCH_MIN_WORDS = 12

def _word_runs(s, n=MATCH_MIN_WORDS):
    """Every run of n consecutive words, lowercased and stripped of punctuation.

    Punctuation goes because a quote pasted into a codebook usually loses or
    gains a comma on the way, and a check defeated by a comma is not a check."""
    w = re.sub(r'[^a-z0-9 ]', ' ', s.lower()).split()
    return {' '.join(w[i:i + n]) for i in range(len(w) - n + 1)}

# Built from RUBRIC rather than from the individual names, so a piece added to
# the codebook later is covered without anyone remembering to add it here.
_shown = ' '.join([RUBRIC['system_prompt'], RUBRIC['facet_rubric'],
                   RUBRIC['schema_hint'],
                   ' '.join(RUBRIC['item_prompts'].values()),
                   ' '.join(d for _n, d in RUBRIC['solo_definitions'].values()),
                   RUBRIC['solo_teaching_case'],
                   RUBRIC['solo_cooccurrence_sentence']])
_shown_runs = _word_runs(_shown)
_leaked = []
for _rid, _it, _t in df[['response_id', 'item_type', 'text']].dropna().itertuples(index=False):
    _hit = _word_runs(str(_t)) & _shown_runs
    if _hit:
        _leaked.append(f'{_rid} [{_it}]: "{sorted(_hit)[0]}"')
assert not _leaked, (
    f'{len(_leaked)} corpus passage(s) share {MATCH_MIN_WORDS} or more consecutive '
    f'words with the text the models are sent, so the codebook would be scoring a '
    f'passage it has already shown the rater: ' + ' | '.join(_leaked[:3]))
print(f'Anti-circularity check: no run of {MATCH_MIN_WORDS} words from any of the '
      f'{len(df):,} corpus responses appears in the {len(_shown):,} characters a '
      f'model is shown.')

Anti-circularity check: no run of 12 words from any of the 2,844 corpus responses appears in the 4,002 characters a model is shown.


### From a reply to a record

The five cells below are everything between "send a prompt" and "have a clean
nine-facet record": the truncation repair, tolerant JSON extraction, a self-test of
both, key normalisation, the failure log, and the step that turns a parsed object
into either a facet record or an explicit unscored row.

**The prefilled-JSON discipline, carried over from Dark Side NB02.** A *prefill* is
the opening of the model's own answer, written for it, which the model then continues.
Anthropic's API allows this; seeding the assistant turn with a single `{` removes the
model's opportunity to open with "Here is the JSON:", because it is already inside the
object. That single character is what makes near-100% parse rates achievable without a
post-processor that guesses. The cost is that the model's reply text does not CONTAIN
its own opening brace, so `parse_json_safe` has to put it back; that is what the
`prefill` argument is for, and passing the wrong value for a leg is the one way to
break parsing for every response at once.

In [16]:
# === REPAIRING A REPLY THAT THE TOKEN LIMIT CUT OFF MID-OBJECT ===
# WHAT THIS FUNCTION DOES: takes the fragment running from the first '{' to the
# END of a reply that will not parse, drops a trailing key/value pair whose value
# was cut in half, closes whatever braces are still open, and tries again.
#
# WHY IT IS WORTH DOING. A reply cut off after six facets still contains six real
# scores. Discarding the whole reply turns six measured facets into nine
# unmeasured ones, and it does so selectively: the replies long enough to be cut
# are the long discursive answers, which are the ones carrying the most facet
# structure.
#
# WHY IT TAKES THE FRAGMENT TO THE END OF THE TEXT RATHER THAN TO THE LAST '}'.
# On a truncated reply the last '}' is not the end of the object; it is the close
# of some nested piece in the middle of it, typically the evidence spans. A
# fragment cut at that brace ends in a half-written structure that no amount of
# closing can rescue, which is the difference between a repair that works and one
# that only looks like it does.
#
# HAZARD, AND THE REASON max_tokens IS GENEROUS: a repaired object is missing
# whatever keys the truncation ate, and coerce_facets leaves those as NaN. NaN is
# skipped by the ensemble mean, so a systematically truncated leg quietly
# contributes fewer raters to the tail facets rather than announcing itself. The
# "mean facets returned" figure in the accounting below is what makes that
# visible: a leg averaging under nine is being cut off, and max_tokens is the
# thing to raise.
def _repair_truncated_json(fragment):
    '''Close off a truncated JSON object. Returns the parsed dict, or None.

    Two attempts, in order. The first keeps the last pair, for a reply cut
    cleanly after a complete value ('"revenge": 0.2'); the second drops it,
    for a reply cut in the middle of one ('"censure": 0.'). Braces are then
    closed by counting how many are still open.

    Counting braces treats a '{' inside a quoted evidence span as if it
    opened an object, so a participant who typed a brace can defeat the
    repair: the reply then comes back unparsed and the row is recorded as
    explicitly unscored, which is the safe direction to fail in.'''
    frag = fragment.rstrip().rstrip(',')
    # The dangling pair: a quoted key, a colon, and a value that stops without
    # ever closing. Anchored to the end of the fragment, so a complete pair
    # earlier in the reply is never touched.
    dropped = re.sub(r',?\s*"[^"]*"\s*:\s*[^,{}\[\]]*$', '', frag)
    for attempt in (frag, dropped):
        attempt = attempt.rstrip().rstrip(',')
        need_close = attempt.count('{') - attempt.count('}')
        if not attempt or need_close < 0:
            continue
        try:
            return json.loads(attempt + '}' * need_close)
        except json.JSONDecodeError:
            continue
    return None

print('_repair_truncated_json defined.')

_repair_truncated_json defined.


In [17]:
# === PARSING: EXTRACT A JSON OBJECT FROM A MODEL REPLY ===
# THE `prefill` ARGUMENT IS NOW ALWAYS ''. Every leg returns a whole object,
# because the Anthropic prefill that used to seed the assistant turn with '{' is
# no longer accepted. The argument survives for two reasons: the self-test below
# still exercises that reply shape, so the recovery path stays proven, and a
# future leg that prefills again would need it. Passing '{' to a leg that does
# not prefill prepends a second brace and breaks parsing for every response on
# that leg at once rather than randomly, which is why the default is ''.
def parse_json_safe(text, prefill=''):
    '''Extract a JSON object from a model reply, in five escalating attempts.

    The reply is not trusted to be clean. In order:
      1. put back a prefilled character if one was used. No current leg
         prefills, so every caller passes '' and this step is a no-op;
         it is kept because the parser is also the thing that would have
         to cope if a leg ever prefilled again;
      2. strip markdown fences, which some models add despite the system
         prompt forbidding them;
      3. parse directly -- with the system prompt doing its job this is
         the normal path and the steps below are for the tail;
      4. fall back to the greedy first-'{'-to-last-'}' slice, which drops
         any prose that leaked in around the object;
      5. last resort, hand everything from the first '{' onwards to
         _repair_truncated_json, which closes off a reply the token limit
         cut in half.

    Returns the parsed dict, or None. None is a RESULT here, not an error:
    it becomes an explicit unscored row rather than a silent omission.'''
    # An empty or None body (a refusal, a dropped connection, an empty content
    # block) is unparseable by definition -- return the None result rather than
    # letting json.loads raise inside a worker thread.
    if not text:
        return None
    # Step 1: put a prefill back, if there was one. Every current leg passes ''
    # so this is a no-op. The signature to remember: a mismatch between what a
    # leg sends and what is passed here fails EVERY response on that leg rather
    # than a random few, so a parse rate near zero on one leg alone points here
    # or at the model string, never at the passages.
    text = prefill + text
    # Step 2: strip ```json fences. The system prompt forbids markdown, and most
    # replies comply, but a fenced object is a trivially recoverable failure and
    # there is no reason to throw the row away over it.
    text = re.sub(r'```(?:json)?\s*', '', text).replace('```', '')
    text = text.strip()
    # Step 3: the clean path. The system prompt alone should carry the
    # overwhelming majority of calls; the code below exists for the tail, not
    # for the norm. If a leg starts leaning on steps 4 and 5, that is a signal
    # about that leg, and the smoke test is where it should show up.
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    first = text.find('{')
    if first == -1:
        # No object anywhere: a refusal, a stray apology, an empty content block.
        return None
    # Step 4: greedy slice. Recovers the object when prose leaked in on either
    # side of it. last > first is checked so an unbalanced fragment does not
    # produce a backwards slice.
    last = text.rfind('}')
    if last > first:
        try:
            return json.loads(text[first:last + 1])
        except json.JSONDecodeError:
            pass
    # Step 5: the truncation repair, on everything from the first '{' to the end.
    return _repair_truncated_json(text[first:])

print('parse_json_safe defined.')

parse_json_safe defined.


In [18]:
# === PROVING THE PARSER BEFORE ANY MONEY IS SPENT ===
# WHAT THIS CELL DOES: runs parse_json_safe over the eight reply shapes that
# actually arrive and stops the notebook if any of them comes out wrong. No API
# call, no cost, no network: it is arithmetic on strings.
#
# WHY IT IS NOT LEFT TO THE LIVE SMOKE TEST BELOW. Twenty paid calls prove the
# parser against whatever the models happen to send that day, which on a good day
# is twenty clean objects and exercises one path out of five. The recovery paths
# -- the markdown fence, the prose preamble, the reply the token limit cut in
# half -- are the ones that matter and the ones a healthy leg never triggers. A
# repair path that has never been exercised is a repair path nobody has ever seen
# work, and it can sit there broken through an entire paid run.
#
# Each case is (name, the reply text as it would arrive, the leg's prefill, what
# must come back). The truncation cases are the point of the cell: 'cut
# mid-number' keeps the two facets that arrived whole, and 'cut mid-span' keeps
# the facet and empties the half-written evidence object.
PARSER_CASES = [
    ('prefilled shape (no leg sends this now)', '"desert":0.8,"revenge":0.2}', '{',
     {'desert': 0.8, 'revenge': 0.2}),
    ('whole object', '{"desert":0.8,"revenge":0.2}', '',
     {'desert': 0.8, 'revenge': 0.2}),
    ('markdown fence', '```json\n{"desert":0.8}\n```', '', {'desert': 0.8}),
    ('prose around it', 'Here is the JSON: {"desert":0.8} Hope that helps.', '',
     {'desert': 0.8}),
    ('cut mid-number', '"desert":0.8,"revenge":0.2,"censure":0.', '{',
     {'desert': 0.8, 'revenge': 0.2}),
    ('cut mid-span', '"desert":0.8,"evidence":{"desert":"he ear', '{',
     {'desert': 0.8, 'evidence': {}}),
    ('a refusal', "I'm sorry, I can't help with that.", '', None),
    ('an empty body', '', '', None),
]
_bad = []
for _name, _raw, _pre, _want in PARSER_CASES:
    _got = parse_json_safe(_raw, prefill=_pre)
    if _got != _want:
        _bad.append(f'{_name}: expected {_want}, got {_got}')
assert not _bad, ('parse_json_safe is not doing what it says it does -- '
                  + ' | '.join(_bad))
print(f'Parser self-test: {len(PARSER_CASES)} reply shapes, every one handled as '
      'expected\n  (prefill, whole object, fence, prose, two truncations, a '
      'refusal, an empty body).')

Parser self-test: 8 reply shapes, every one handled as expected
  (prefill, whole object, fence, prose, two truncations, a refusal, an empty body).


In [19]:
# === NORMALISING THE KEYS A MODEL SENDS BACK ===
def normalize_label(s):
    '''Map whatever key a model returned back to a canonical facet name.

    Models paraphrase keys even under a literal schema: 'Moral Balance',
    'moral-balance', 'desert - the offender earned it'. Three attempts, in
    order of strictness: exact match after lowercasing and folding '_' and
    '-' to spaces; then the prefix before ' - '; then a substring match.
    Returns None when nothing matches, and the caller drops the key rather
    than guessing -- a mis-mapped key would write one facet's score into
    another facet's column, which is worse than a missing value.

    Re-tooled from the Dark Side helper to the nine FACETS.'''
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    s = str(s).strip().lower().replace('_', ' ').replace('-', ' ')
    if not s:
        return None
    s = ' '.join(s.split())
    # Built from FACETS, so the accepted vocabulary can never drift away from
    # the canonical list declared above.
    canon = {f.replace('_', ' '): f for f in FACETS}
    # Strictest first: an exact hit on the space-folded name.
    if s in canon:
        return canon[s]
    # Then 'desert - the offender earned it' style keys, where the model echoed
    # part of the definition into the key.
    if ' - ' in s:
        prefix = s.split(' - ')[0].strip()
        if prefix in canon:
            return canon[prefix]
    # Loosest: the canonical name appears somewhere inside the returned key.
    # Safe here only because the nine names are mutually non-containing, so no
    # key can match two facets.
    for k, f in canon.items():
        if k in s:
            return f
    return None

print('normalize_label defined.')

normalize_label defined.


In [20]:
# === FAILURE ACCOUNTING ===
# FAILURE_LOG holds the VERBATIM text of the first few replies that could not be
# parsed, capped at 5. Small on purpose: it is a diagnostic sample, not an
# archive, and the cap keeps a systematic failure -- say, one leg refusing every
# graphic passage -- from filling the output with thousands of copies of the same
# message.
#
# It answers the question the counts cannot: WHY did parsing fail. Prose
# preamble, a refusal, a fence, a truncation and an authentication error all
# arrive as the same None, and they call for completely different fixes. The log
# is printed at the end of this notebook and is the first thing to read when a
# leg's parse rate comes back low.
#
# The counts themselves are never hidden: every parse failure is an explicitly
# unscored row in smoke_test_results.csv regardless of whether it made it into
# this log.
FAILURE_LOG = []

# Records the model key, the first 120 characters of the prompt (enough to
# identify the passage) and 500 characters of the raw reply. `last_raw` is None
# when the failure was an exception rather than an unparseable body, and those
# are skipped -- there is no text to inspect.
def _log_failure(model, prompt, last_raw):
    if len(FAILURE_LOG) < 5 and last_raw is not None:
        FAILURE_LOG.append({'model': model, 'prompt_first120': prompt[:120],
                            'raw_response': last_raw[:500]})

# OFF_SCALE is the other kind of failure, and it is the dangerous kind, because
# nothing about it looks like a failure. A reply that parses perfectly and
# returns 7 for every facet is a rater answering on a 0-10 scale, and every count
# in this notebook would read it as a success. Uncapped, unlike FAILURE_LOG:
# these are rare by construction, and if they are not rare then the number itself
# is the finding.
OFF_SCALE = []

def _log_off_scale(model, where, what):
    OFF_SCALE.append({'model': model, 'where': where, 'value': what})

print('Failure log armed (raw text of the first 5 unparseable replies), and the '
      'off-scale log armed (any facet value outside '
      f'{FACET_MIN}-{FACET_MAX}).')

# === PERMANENT VERSUS TRANSIENT ===============================================
# The retry loop needs to answer two different questions. "Wait and try again?"
# is the rate-limit test. This is the opposite one: "will this EVER succeed?" A
# 4xx that is not a 429 is a statement about the request, not about the server's
# mood. Matching on the status code rather than on an exception class keeps this
# working across both client libraries in this pipeline and across SDK versions.
def _is_permanent(e):
    """True when retrying cannot possibly help.

    Two kinds qualify. A 4xx that is not a 429 is a statement about the
    REQUEST -- a model id that does not exist, a parameter this model no
    longer accepts, a key that is not valid. And a TypeError or
    AttributeError is a malformed CALL: no request ever left the machine,
    so there is nothing for the network to do differently next time.

    The second kind is not hypothetical. anthropic 1.0.0 removed the
    temperature parameter, and every Claude call died with
    TypeError: Messages.create() got an unexpected keyword argument
    'temperature' -- retried five times per row by the generic handler and
    then filed as an unparsed row, which is how it surfaced as "0/20
    parsed" rather than as the one sentence it actually was."""
    if isinstance(e, (TypeError, AttributeError)):
        return True
    code = getattr(e, 'status_code', None) or getattr(
        getattr(e, 'response', None), 'status_code', None)
    return isinstance(code, int) and 400 <= code < 500 and code != 429

print('_is_permanent defined (a 4xx that is not 429, or a malformed call, '
      'is never retried).')

Failure log armed (raw text of the first 5 unparseable replies), and the off-scale log armed (any facet value outside 0.0-1.0).
_is_permanent defined (a 4xx that is not 429, or a malformed call, is never retried).


In [21]:
# === FROM A PARSED OBJECT TO A CLEAN NINE-FACET RECORD ===
def coerce_facets(parsed):
    '''Turn one parsed reply into a clean nine-facet + holistic record, or None.

    Starts from all-NaN over FACETS and fills only keys that normalize to a
    canonical facet, so an extra key the model invented is dropped and a
    facet the model omitted stays NaN (unmeasured) rather than becoming 0.0
    (measured absent). The distinction matters: the ensemble mean skips NaN
    but averages in a 0.0.

    parsed is None -> return None, which becomes an explicit unscored row.
    Nothing here ever drops a response.'''
    if parsed is None:
        return None
    # NaN, not 0.0. An unreturned facet is UNMEASURED; a returned 0.0 is
    # MEASURED ABSENT. Collapsing the two would let a truncated or partial reply
    # add fake zeros to the ensemble mean and drag every facet toward the floor.
    out = {f: np.nan for f in FACETS}
    # Whatever came back off the 0.0-1.0 scale, kept as text so the caller can
    # report the value the model actually sent rather than the fact that one
    # existed.
    off_scale = []
    for k, v in parsed.items():
        canon = normalize_label(k)
        if canon is None:
            continue
        try:
            val = float(v)
        except (ValueError, TypeError):
            out[canon] = np.nan
            continue
        # THE SCALE CHECK. A model told to "score from 0.0 to 1.0" can answer on
        # a 0-10 or a 1-7 scale, and that is the one fault every other number
        # here is blind to: the reply parses, all nine facets come back, the
        # parse rate reads 20/20 and every value is wrong. An out-of-range value
        # is NOT clipped -- clipping 7 to 1.0 would invent a measurement nobody
        # made -- it is recorded as unmeasured and reported, loudly, below.
        if FACET_MIN <= val <= FACET_MAX:
            out[canon] = val
        else:
            off_scale.append(f'{canon}={v}')
            out[canon] = np.nan
    # The holistic call is validated against the four allowed levels and set to
    # None otherwise, so Krippendorff's alpha downstream sees a clean category
    # variable and an invented fifth category cannot become its own level.
    #
    # `evidence` is kept only when it is a dict. It is optional by design (see
    # the schema cell) and is carried for auditing; nothing downstream scores it,
    # so a malformed evidence field can never affect a facet value.
    hol = str(parsed.get('holistic', '')).strip().lower()
    out['holistic'] = hol if hol in HOLISTIC_LEVELS else None
    out['evidence'] = parsed.get('evidence', {}) if isinstance(parsed.get('evidence'), dict) else {}
    out['off_scale'] = off_scale
    return out

print('coerce_facets defined (a missing facet stays unmeasured, never a zero; a '
      f'value outside {FACET_MIN}-{FACET_MAX} is unmeasured and reported).')

coerce_facets defined (a missing facet stays unmeasured, never a zero; a value outside 0.0-1.0 is unmeasured and reported).


### The raters

Three models, from three developers, one of them open-weight. This is the panel the
rubric is proven against. A leg whose key is not configured is reported as **not run**
and the other legs still get tested -- finding out here that one key of three is
missing costs a minute, and finding out in hour two of the paid run costs the run.

In [22]:
# === API KEYS AND CLIENTS ===
# KEYS COME FROM COLAB SECRETS, NEVER FROM A CELL
#   A key pasted into a cell is saved inside the .ipynb, and this .ipynb goes to
#   OSF with the prompts and the raw output (prereg 6.7). Secrets keep the key
#   out of the file. userdata.get returns '' (not an error) when the secret
#   exists but is blank, and a blank key produces authentication failures that
#   look exactly like rate limits, so an empty value is treated as "not
#   configured" rather than as a key.
#
# A MISSING KEY DOES NOT STOP THIS NOTEBOOK. It marks that leg not run, prints
# one line, and the notebook carries on with the legs that are configured. That
# is the whole point of a smoke test: the two legs that ARE configured still get
# proven, and the missing one is named in plain words instead of arriving as a
# wall of authentication errors that the retry loop swallows into unscored rows
# halfway through a paid corpus pass.
def _load_secret(name):
    '''Return the key, or None if it is not configured. Never raises.'''
    val = None
    try:
        if userdata is not None:
            val = userdata.get(name)
    except Exception as e:
        print(f'  {name}: not readable from Colab Secrets ({type(e).__name__}).')
    if not val:
        val = os.environ.get(name)          # fallback when not running in Colab
    if not val:
        print(f'  {name}: NOT CONFIGURED. Add it via the key icon in the left sidebar.')
        return None
    # Last four characters only -- enough to tell two keys apart in the run log,
    # not enough to be a key.
    print(f'  {name}: loaded (ends in ...{val[-4:]})')
    return val

# One key per DEVELOPER, because the ensemble's independence is at the developer
# level, not the model level. Two snapshots from one lab share pretraining data,
# tokenizer, post-training policy and refusal behaviour, so they share blind
# spots; averaging them is closer to sampling one rater twice than to adding a
# rater. Prereg 6.7's ">=3 developers" is what makes the inter-model ICC a
# reliability coefficient rather than a self-consistency check.
print('Loading API keys...')
KEY_CLAUDE = _load_secret('Punishment_ClaudeAPI')
KEY_GPT    = _load_secret('Punishment_OpenAIAPI')       # second-developer leg
KEY_OPEN   = _load_secret('Punishment_OpenWeightAPI')   # open-weight leg

# One client per leg, built only where the key exists.
#
# Leg 3 is the OPEN-WEIGHT model and it is not interchangeable with the other
# two. Prereg 6.7 requires it: it is the architecturally most different rater in
# the set, it is fully reproducible from published weights by anyone holding the
# OSF bundle, and its disagreement with the two proprietary models is treated as
# signal about a shared blind spot rather than noise to be majority-voted away.
# It is reached through an OpenAI-compatible endpoint, so the same adapter serves
# legs 2 and 3 and only base_url and the model string move. Together is shown;
# swap base_url for another host if the team prefers one.
claude_client = anthropic.Anthropic(api_key=KEY_CLAUDE) if KEY_CLAUDE else None
gpt_client    = OpenAI(api_key=KEY_GPT) if KEY_GPT else None
open_client   = OpenAI(api_key=KEY_OPEN,
                       base_url='https://api.together.xyz/v1') if KEY_OPEN else None

Loading API keys...
  Punishment_ClaudeAPI: loaded (ends in ...sAAA)
  Punishment_OpenAIAPI: loaded (ends in ...yeUA)
  Punishment_OpenWeightAPI: loaded (ends in ..._DZU)


In [23]:
# === THE MODEL REGISTRY ***PLACEHOLDERS -- PIN DATED VERSION STRINGS*** ===
# This dict is the rater panel. Everything downstream -- the ensemble mean, the
# ICC, the holistic majority, the shared-error diagnostic, the per-model columns
# in the export -- iterates over these keys, so the roster is a measurement
# decision and not configuration.
#
# THE THREE CONSTRAINTS, AND WHAT EACH ONE IS FOR (prereg 6.7; design section 4)
#   >=3 MODELS.       Two raters give an ICC but no majority: the holistic call
#                     has no tie-break, and one outlier moves the mean by half
#                     its distance. Three is the smallest panel where a single
#                     odd reading is visible AS an odd reading. A fourth is
#                     optional and mainly steadies the ICC on the sparse facets.
#   >=3 DEVELOPERS.   The independence that matters is at the lab level, not the
#                     checkpoint level. Three snapshots from one developer would
#                     post a high ICC and certify nothing.
#   >=1 OPEN-WEIGHT.  Reproducibility (anyone can re-run the leg from weights)
#                     and diversity (a different pretraining mix and a different
#                     post-training regime, so its errors have the best chance of
#                     being uncorrelated with the proprietary pair).
#
# WHY A STRONGER ANTHROPIC TIER THAN THE DARK SIDE PIPELINE USED. Dark Side NB02
# scored a prosocial/dark binary with Haiku. Here the scorer has to hold desert
# apart from proportionality apart from censure apart from moral balance -- four
# boundaries that a coarse valence read collapses (design section 1). Cost is
# trivial for a one-time batch of about 900 responses x 3 models.
#
# PIN EXACT DATED STRINGS BEFORE THE PRODUCTION RUN. An alias ending in "-latest"
# silently re-points at a new checkpoint, which makes the run unrepeatable and
# quietly invalidates the pinned-version clause of prereg 6.7. Confirm the IDs at
# run time: the Dark Side strings are about a year old. See MEASUREMENT_DESIGN.md
# section 11, decision 1, for the open roster choices.
# NOT ONE OF THE THREE LEGS CARRIES A DATE SUFFIX, AND THAT IS NOT AN OVERSIGHT.
# Prereg 6.7 asks for a "pinned dated version string", which in the Dark Side era
# meant a form like 'claude-3-5-sonnet-20241022'. All three providers have since
# moved their current lines off dated snapshots:
#   claude-opus-4-6                          IS the complete Anthropic id.
#                                            Appending a date produces a 404.
#   gpt-5.6-terra                            is the explicit tier id. The BARE
#                                            alias 'gpt-5.6' routes to sol and is
#                                            a moving target -- never use it here.
#   meta-llama/Llama-3.3-70B-Instruct-Turbo  is the whole Together string.
# What each of these IS is explicit and non-routing, which is the property the
# clause was protecting: none is a '-latest' tag that silently re-points. The
# rest of the guarantee comes from the archive -- notebook 06 writes every model
# reply verbatim to llm_scores_<MODEL>.jsonl -- so a run stays auditable from its
# own output even if a provider later moves an id. Record this in the OSF
# deviation register: it is a change in what the industry offers, not a
# loosening on our side.
#
# WHY THIS PANEL, AND WHY EVERY LEG RUNS GREEDY.
# Prereg 6.7 fixes the sampling setting, and this study operationalised that as
# temperature 0. Holding that across all three legs is what drove each choice:
#   claude-opus-4-6  the strongest Anthropic tier that still ACCEPTS temperature.
#                    It was removed from Opus 4.7 onward, so claude-opus-5 would
#                    run leg 1 at the model default while legs 2 and 3 stayed
#                    greedy. The extra variance on one leg alone would surface as
#                    inter-model disagreement and depress the ICC -- reading as
#                    rater disagreement when it is really sampling noise.
#   gpt-5.6-terra    with reasoning_effort='none'. GPT-5.x accepts temperature
#                    ONLY at that setting; at any real reasoning effort it
#                    rejects it outright. The classic GPT-4.x models that took
#                    temperature unconditionally are being retired (gpt-4o
#                    2024-05-13 is already gone), so this is how this leg stays
#                    greedy, not a preference.
#   Llama-3.3-70B    takes temperature 0 unconditionally. Meta-developed, so it
#                    supplies the third developer AND the open-weight leg in one.
# The cost of this panel is that both proprietary legs run deliberately de-tuned:
# an Anthropic tier one generation back, and an OpenAI model with its reasoning
# switched off. That is the price of the registered fixed-setting clause, and it
# was paid deliberately.
MODELS_ALL = {
    'claude': 'claude-opus-4-6',                          # Anthropic. Takes temperature.
    'gpt':    'gpt-5.6-terra',                            # OpenAI. Needs effort='none'.
    'open':   'meta-llama/Llama-3.3-70B-Instruct-Turbo',  # Meta, open weights, Together.
}
_CLIENT_FOR = {'claude': claude_client, 'gpt': gpt_client, 'open': open_client}
# MODELS is the roster this notebook will actually exercise. A leg with no key is
# dropped here and named as NOT RUN below -- never left in the roster, because a
# roster key with no client is an error inside the scoring call rather than a
# reportable absence.
MODELS  = {k: v for k, v in MODELS_ALL.items() if _CLIENT_FOR[k] is not None}
NOT_RUN = [k for k in MODELS_ALL if k not in MODELS]

print('Rater panel:')
for k, v in MODELS_ALL.items():
    state = 'will run' if k in MODELS else 'NOT RUN (no key)'
    print(f'  {k:8s} -> {v:52s} {state}')
if NOT_RUN:
    print(f'\n  {len(NOT_RUN)} of {len(MODELS_ALL)} legs are NOT RUN: ' + ', '.join(NOT_RUN))
    print('  The rubric is still written and the configured legs are still tested.')
# A PLACEHOLDER IS TREATED EXACTLY LIKE A MISSING KEY: the leg is dropped from
# the roster and named, and no call is ever made on it.
#
# THIS USED TO BE A WARNING, AND THE WARNING WAS THE BUG. The first run of this
# notebook printed the words below, carried on, and made sixty calls that every
# one of them failed with model-not-found -- reported as "0/20 parsed" on all
# three legs, with the actual error discarded by the retry loop. A smoke test
# that spends calls on a roster it has already diagnosed as unrunnable is not
# checking anything; it is only paying to rediscover what it printed. Dropping
# the leg here means the configured legs still get proven, which is the whole
# point, and the unpinned ones cost nothing.
_placeholder = [k for k in MODELS if '<' in MODELS[k]]
for k in _placeholder:
    del MODELS[k]
NOT_PINNED = _placeholder
# NOT_RUN is recomputed here, AFTER the placeholder legs are removed, so that it
# means "did not run, for any reason" rather than only "had no key". rubric.json
# below writes legs_run and not_run as the record of what happened; computed
# before this line, an unpinned leg would appear in NEITHER and would vanish from
# the record silently. Naming a leg is the whole convention of this notebook.
NOT_RUN = [k for k in MODELS_ALL if k not in MODELS]
if NOT_PINNED:
    print('\n  NOT PINNED, so NOT RUN: ' + ', '.join(NOT_PINNED))
    print('  Those strings still contain <angle brackets>. Pin the exact snapshot')
    print('  for each, then re-run this cell. No call was made on them.')

# The prereg constraint is checked against the roster that will ACTUALLY run, not
# against the roster as declared. A panel that is short a leg is fine here -- a
# smoke test proves plumbing -- but it must not be mistaken for the registered
# design, so the shortfall is stated in the notebook that would otherwise imply
# the panel is complete.
if len(MODELS) < 3:
    print(f'\n  ROSTER INCOMPLETE: {len(MODELS)} of 3 legs will run.')
    print('  Prereg 6.7 requires >=3 models from >=3 developers, >=1 open-weight.')
    print('  That is a requirement on the PAID pass in notebook 06, not on this')
    print('  smoke test, which is free to prove one leg at a time. Notebook 06')
    print('  refuses to start on an unpinned model.')
if not MODELS:
    raise SystemExit(
        'No leg can run: every configured leg is either missing its key or '
        'still holds a placeholder model string. Fix one of those and re-run. '
        'Nothing was spent.')

Rater panel:
  claude   -> claude-opus-4-6                                      will run
  gpt      -> gpt-5.6-terra                                        will run
  open     -> meta-llama/Llama-3.3-70B-Instruct-Turbo              will run


### The three call adapters

Three developers, three software shapes, one prompt. Each adapter does the same four
things -- send the SAME prompt at the same temperature, read the text out of whatever
envelope that provider uses, parse it, retry -- so the only thing that varies across
legs is the model. Anything richer, a leg-specific reformat or a leg-specific repair,
would put prompt variance into the agreement statistics.

In [24]:
# === LEG 1: ANTHROPIC ===
# The retry shape is carried over from Dark Side NB02 and is deliberately
# asymmetric, because the two failure kinds want opposite treatment:
#   parse failure   a short fixed pause (0.5s) and try again. At temperature 0
#                   the retry is near-identical, so this mostly buys a second
#                   chance at a transport-level truncation rather than a
#                   different reading.
#   rate limit      exponential backoff 2**attempt (1, 2, 4, 8, 16s). Retrying a
#                   throttle immediately deepens the queue for every worker at
#                   once and turns a brief throttle into a cascade.
#   other error     one second, and give up on the final attempt.
# max_retries=4 means up to five attempts. Beyond that the failure is structural,
# not transient, and burning more calls on it only inflates cost.
#
# ALL THREE ADAPTERS READ GEN_KW rather than carrying their own copy of the
# numbers in a signature default. Otherwise the settings cell above prints one
# temperature, the calls go out at another, and rubric.json records a third --
# and a smoke test that cannot prove a setting change is not proving the run.
def claude_call(prompt, model_id, max_retries=4, return_raw=False):
    '''One Claude call.

    Returns the parsed dict, or None after the retry budget is spent. It
    never raises: in the paid run this is handed to a pool of worker
    threads, and an exception escaping here would surface only when the
    result was read, taking the row's identity with it. None is the honest,
    recorded outcome.

    return_raw=True returns (raw_reply_text, parsed) instead of parsed. The
    smoke test is the only caller that asks for it, and it asks because
    showing a human what the model actually said beside what the parser made
    of it is the entire job of a smoke test. Reading the raw text out of THIS
    call path, rather than out of a second parallel copy of it, is what keeps
    the smoke test testing the code that will really run.'''
    last_raw = None
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            response = claude_client.messages.create(
                model=model_id,
                max_tokens=GEN_KW['max_tokens'],
                temperature=GEN_KW['temperature'],
                system=JSON_SYSTEM_PROMPT,
                # NO PREFILLED ASSISTANT TURN, AND THAT IS NOT OPTIONAL.
                # Dark Side NB02 seeded the assistant turn with '{' so the model
                # resumed inside the object with no room for a preamble. Current
                # Anthropic models REJECT a trailing assistant turn outright with
                # a 400, so the technique that used to buy near-100% parse rates
                # now buys a failed call on every single response. What replaces
                # it is JSON_SYSTEM_PROMPT (which already forbids preamble,
                # fences and commentary) plus the five escalating recovery steps
                # in parse_json_safe. The smoke test below MEASURES whether that
                # is enough: if this leg comes back short of 20/20 while the
                # OpenAI-compatible legs are clean, the next step is
                # output_config.format with an explicit json_schema, which
                # enforces the shape rather than asking for it.
                messages=[{'role': 'user', 'content': prompt}],
            )
            content = response.content[0].text
            last_raw = content
            # prefill='' now, to match: the reply carries its own opening brace.
            # Passing '{' here would prepend a second one and break parsing for
            # every response on this leg at once.
            parsed = parse_json_safe(content, prefill='')
            if parsed is not None:
                return (last_raw, parsed) if return_raw else parsed
            time.sleep(0.5)
            # A parse failure is not an exception -- pause briefly and let the
            # loop try again with the same budget.
        except anthropic.RateLimitError as e:
            last_err = repr(e)[:200]
            time.sleep(2 ** attempt)
        except anthropic.APIStatusError as e:
            # A PERMANENT ERROR IS NOT RETRIED, AND ITS MESSAGE IS NOT DISCARDED.
            # 404 model-not-found, 400 bad-request, 401 bad-key: none of these
            # get better on the fifth attempt. The old code sent all of them
            # through the generic handler, slept, tried again, and finally
            # reported the row as unparsed -- so three separate fatal faults (an
            # unpinned model string, the rejected prefill, a removed sampling
            # parameter) all presented as the same "0/20 parsed" with no way to
            # tell them apart. Breaking immediately makes the API's own sentence
            # the thing the reader sees.
            last_err = f'HTTP {e.status_code}: {getattr(e, "message", str(e))}'[:300]
            break
        except Exception as e:
            last_err = repr(e)[:200]
            # A malformed call (TypeError, AttributeError) never left the
            # machine and will fail identically on every retry -- stop now and
            # let the message be the finding.
            if _is_permanent(e):
                break
            if attempt == max_retries:
                break
            time.sleep(1)
    _log_failure('claude', prompt, last_raw)
    if return_raw:
        # Always a string, so the smoke test has something to show either way.
        # The angle brackets mark the line as ours and not the model's.
        return (last_raw if last_raw is not None
                else f'<<no reply text came back; last error: {last_err}>>', None)
    return None

print('claude_call defined (leg 1, bare JSON object, no prefill).')

claude_call defined (leg 1, bare JSON object, no prefill).


In [25]:
# === LEG 2: THE SECOND DEVELOPER, OPENAI-COMPATIBLE ===
# Same prompt, same temperature, same retry budget; only the envelope differs.
# Two provider-specific affordances are used where they exist, and neither
# changes what is asked:
#   seed=SEED                   best-effort decoding determinism on top of
#                               temperature 0 (the DECODING seed, unrelated to
#                               the frozen SEED half of the corpus)
#   response_format json_object server-side JSON enforcement. This is the only
#                               leg with a hard guarantee of syntactic validity;
#                               the other two rely on the system prompt and on
#                               parse_json_safe, so if any leg is going to post
#                               a parse failure it will not be this one
# prefill='' here, as on every leg: this one returns a complete object.
#
# The error handler matches on the word 'rate' rather than catching a typed
# error, because an OpenAI-compatible shim in front of a third-party host does
# not always raise the library's own rate-limit class.
def gpt_call(prompt, model_id, max_retries=4, return_raw=False):
    last_raw = None
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            resp = gpt_client.chat.completions.create(
                model=model_id,
                temperature=GEN_KW['temperature'],
                # THIS IS WHAT MAKES temperature LEGAL ON THIS LEG. GPT-5.x
                # rejects temperature at any real reasoning effort; at 'none' it
                # behaves as a classic model and accepts it. Sent explicitly
                # rather than relied on as a default, because the default has
                # moved between GPT-5 releases and a silent change here would
                # un-fix the setting prereg 6.7 fixes.
                reasoning_effort=GPT_REASONING_EFFORT,
                seed=SEED,
                # max_completion_tokens, NOT max_tokens. The GPT-5.x family
                # rejects max_tokens outright ("Unsupported parameter") because
                # it does not account for reasoning tokens. Same value out of
                # GEN_KW, different name on this provider: the setting has not
                # changed, only what this endpoint calls it.
                max_completion_tokens=GEN_KW['max_tokens'],
                messages=[
                    {'role': 'system', 'content': JSON_SYSTEM_PROMPT},
                    {'role': 'user', 'content': prompt},
                ],
                # Server-side JSON mode: the provider guarantees a
                # syntactically valid object. No other leg has this.
                response_format={'type': 'json_object'},
            )
            content = resp.choices[0].message.content
            last_raw = content
            parsed = parse_json_safe(content, prefill='')  # no prefill; the parser handles it
            if parsed is not None:
                return (last_raw, parsed) if return_raw else parsed
            time.sleep(0.5)
        except Exception as e:
            last_err = repr(e)[:200]
            # _is_permanent is tested FIRST. It returns False for a 429, so
            # rate limits still fall through to the backoff below, but a 4xx or
            # a malformed call is caught even when its message happens to
            # contain the word "rate" -- e.g. a 400 naming a bad parameter.
            if _is_permanent(e):
                # NEVER RETRIED. A 4xx that is not a 429 is a statement about the
                # REQUEST -- a model id that does not exist, a parameter this
                # model no longer accepts, a key that is not valid. Retrying buys
                # four more copies of the same refusal, and the old code then
                # reported the row as unparsed with the API's own sentence thrown
                # away. That is exactly how an unpinned roster came back as
                # "0/20 parsed" instead of as "model not found".
                last_err = f'permanent, not retried: {e!r}'[:300]
                break
            elif 'rate' in str(e).lower():
                time.sleep(2 ** attempt)
            elif attempt == max_retries:
                break
            else:
                time.sleep(1)
    _log_failure('gpt', prompt, last_raw)
    if return_raw:
        return (last_raw if last_raw is not None
                else f'<<no reply text came back; last error: {last_err}>>', None)
    return None

print('gpt_call defined (leg 2, server-side JSON mode).')

gpt_call defined (leg 2, server-side JSON mode).


In [26]:
# === LEG 3: THE OPEN-WEIGHT MODEL, HOSTED OPENAI-COMPATIBLE ===
# Identical to leg 2 except that response_format is NOT requested: hosted
# open-weight endpoints vary in whether they support JSON mode, and asking for an
# unsupported option is an error on some providers. parse_json_safe absorbs the
# difference, which is exactly why the parser is written to be tolerant.
#
# This leg is the one prereg 6.7 requires and the one design section 4 relies on
# for architectural diversity. Its scores stay individually addressable all the
# way to the export, so its standalone verdict can be reported on its own line
# rather than being averaged into the consensus it is meant to check.
def open_call(prompt, model_id, max_retries=4, return_raw=False):
    last_raw = None
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            resp = open_client.chat.completions.create(
                model=model_id,
                temperature=GEN_KW['temperature'],
                seed=SEED,
                max_tokens=GEN_KW['max_tokens'],
                messages=[
                    {'role': 'system', 'content': JSON_SYSTEM_PROMPT},
                    {'role': 'user', 'content': prompt},
                ],
            )
            content = resp.choices[0].message.content
            last_raw = content
            parsed = parse_json_safe(content, prefill='')
            if parsed is not None:
                return (last_raw, parsed) if return_raw else parsed
            time.sleep(0.5)
        except Exception as e:
            last_err = repr(e)[:200]
            # _is_permanent is tested FIRST. It returns False for a 429, so
            # rate limits still fall through to the backoff below, but a 4xx or
            # a malformed call is caught even when its message happens to
            # contain the word "rate" -- e.g. a 400 naming a bad parameter.
            if _is_permanent(e):
                # NEVER RETRIED. A 4xx that is not a 429 is a statement about the
                # REQUEST -- a model id that does not exist, a parameter this
                # model no longer accepts, a key that is not valid. Retrying buys
                # four more copies of the same refusal, and the old code then
                # reported the row as unparsed with the API's own sentence thrown
                # away. That is exactly how an unpinned roster came back as
                # "0/20 parsed" instead of as "model not found".
                last_err = f'permanent, not retried: {e!r}'[:300]
                break
            elif 'rate' in str(e).lower():
                time.sleep(2 ** attempt)
            elif attempt == max_retries:
                break
            else:
                time.sleep(1)
    _log_failure('open', prompt, last_raw)
    if return_raw:
        return (last_raw if last_raw is not None
                else f'<<no reply text came back; last error: {last_err}>>', None)
    return None

# --- Dispatch table: model key -> its adapter ------------------------------
# Keeps the scoring loop free of any per-leg branching, so adding a fourth rater
# is two lines (an adapter here, an entry in the roster) and touches no analysis
# code. Swapping the second-developer leg to Gemini means adding a 'gemini'
# adapter that sends the SAME prompt and registering it in the roster.
#
# HAZARD: the keys here and the keys in the roster must match exactly. The
# scorers look the adapter up as CALL[model_key] before any adapter's own error
# handling is reached, so a roster key with no CALL entry raises immediately --
# the run stops rather than quietly returning a leg of nothing, which is the
# behaviour to want here.
CALL = {'claude': claude_call, 'gpt': gpt_call, 'open': open_call}
print('open_call defined (leg 3, no JSON mode requested). Dispatch table: '
      + ', '.join(CALL))

open_call defined (leg 3, no JSON mode requested). Dispatch table: claude, gpt, open


In [27]:
# === THE TWO SCORERS THE PAID RUN CALLS ===
def score_one(text, item_type, model_key, return_raw=False):
    '''Score one passage on one model: the full nine-facet joint call.

    Re-applies the word floor even though the corpus filter already enforced
    it, so a short text arriving by any route returns None instead of a
    fabricated score. In the paid run this is the function handed to the
    thread pool, and its None is what becomes an explicit unscored row.'''
    if not isinstance(text, str) or len(text.split()) < MIN_WORDS:
        return (None, None) if return_raw else None
    prompt = build_user_prompt(text, item_type)
    out = CALL[model_key](prompt, MODELS[model_key], return_raw=return_raw)
    raw, parsed = out if return_raw else (None, out)
    rec = coerce_facets(parsed)
    # Anything the model returned off the 0.0-1.0 scale is logged against the leg
    # that sent it, because the fix differs by leg: one model reading the
    # instruction as 0-10 is a prompt problem, all three doing it is a codebook
    # problem.
    if rec is not None:
        for _item in rec['off_scale']:
            _log_off_scale(model_key, 'joint', _item)
    return (raw, rec) if return_raw else rec

def score_facet_solo(text, item_type, facet, model_key, mixed_anchor=True, return_raw=False):
    '''Score ONE facet on ONE model in its own call. Returns a float or None.

    Same word floor as score_one, and the same None-not-drop rule. The reply
    is a one-key object, but the key still goes through normalize_label,
    because a model that answers a capitalised key is right and would
    otherwise be discarded as a failure.'''
    if not isinstance(text, str) or len(text.split()) < MIN_WORDS:
        return (None, None) if return_raw else None
    prompt = build_solo_prompt(text, item_type, facet, mixed_anchor=mixed_anchor)
    # The same three adapters, the same dispatch table, the same temperature and
    # retry budget as the joint call. Only the PROMPT differs -- which is the
    # point: any change in the resulting correlation is attributable to the
    # isolation and to nothing else.
    out = CALL[model_key](prompt, MODELS[model_key], return_raw=return_raw)
    raw, parsed = out if return_raw else (None, out)
    # Pull the single value out by canonical name rather than by position, and
    # return None if the key never matched or the value will not cast to float.
    # A None here is an unscored cell that the split-call mean skips -- it never
    # becomes a 0.0, which would read as "no revenge expressed" and bias the very
    # correlation this pass exists to estimate honestly.
    val = None
    if parsed is not None:
        for k, v in parsed.items():
            if normalize_label(k) == facet:
                val = v
                break
        try:
            val = float(val) if val is not None else None
        except (ValueError, TypeError):
            val = None
        # The same scale check the joint call applies. A solo value off the scale
        # is not a score, and it must not reach the E4 split-call estimate, which
        # is the estimate carrying the primary weight.
        if val is not None and not (FACET_MIN <= val <= FACET_MAX):
            _log_off_scale(model_key, f'split/{facet}', f'{facet}={val}')
            val = None
    return (raw, val) if return_raw else val

# Still no API calls made. Defining these costs nothing.
print('score_one and score_facet_solo defined. No calls sent yet.')

score_one and score_facet_solo defined. No calls sent yet.


### The twenty-row smoke test

Now the money gets spent, all two cents of it. Twenty real responses go to every
configured leg, and the raw reply is printed next to the parsed record so a prompt or
parsing fault is visible here rather than after a full paid run.

In [28]:
# === PICK THE TWENTY ROWS ===
# THIS NOTEBOOK IGNORES RUN_MODE. Twenty rows is not a mode here, it is the
# entire job: the point is to spend two cents proving the rubric before the paid
# pass spends the budget. (done() stamps RUN_MODE into the manifest for every
# file every notebook writes, so it appears beside these two outputs as well.
# rubric.json is byte-identical either way, and the note recorded with it says
# so.)
SMOKE_N    = 20      # responses per leg
SHOW_RAW_N = 3       # how many replies per leg are printed in full below. Every
                     # FAILURE is printed in full regardless of this number,
                     # because a failure is the thing you actually need to read.

# All three item types must be exercised. build_user_prompt injects a different
# question per item_type, and a typo in one of the three would otherwise show up
# only in the paid run -- and then only as one item scoring oddly, which reads
# like a finding rather than like a bug.
PER_ITEM = {'backward': 7, 'forward': 7, 'ranking': 6}
assert sum(PER_ITEM.values()) == SMOKE_N, 'PER_ITEM must add up to SMOKE_N'

# What the draw can actually deliver. If an item type has fewer scorable rows
# than its quota -- because a column did not coalesce upstream, or because every
# answer to that item fell below the word floor -- the shortfall is a fact about
# the corpus, and the next cell names it rather than letting it pass as a full
# draw.
POOLS = {it: list(df_valid.index[df_valid['item_type'] == it]) for it in PER_ITEM}
EXPECTED_DRAW = {it: min(k, len(POOLS[it])) for it, k in PER_ITEM.items()}

# Drawn with RANDOM_SEED so the same twenty rows come back on every re-run. A
# smoke test on a fresh random draw each time cannot tell a fixed bug from an
# unlucky passage.
pick_idx = []
for _it, _k in PER_ITEM.items():
    pick_idx += list(pd.Series(POOLS[_it]).sample(n=EXPECTED_DRAW[_it],
                                                  random_state=RANDOM_SEED))

# Force the single longest response into the draw. NOT because a long passage
# risks a truncated reply -- the reply is a fixed nine-key object and the passage
# is cut to TEXT_TRUNCATION_CHARS before it is sent, so a 900-word answer and a
# 60-word one produce replies of the same size -- but because the longest row is
# the one where the passage window actually bites, and it is worth two cents to
# watch a model score a passage that arrives cut off mid-sentence.
#
# The row it replaces is one of the SAME item type, so the per-item counts
# printed below are the counts that were really drawn. Overwriting the last slot
# regardless, which is the obvious way to write this, spends a ranking slot on a
# backward row whenever the longest answer happens to be a backward one, and the
# printed plan then describes a draw that never happened.
#
# Guarded on a non-empty corpus, because a corpus with nothing scorable in it at
# all has no longest row to ask for, and "which row is longest" is not the error
# a reader would need to see in that case. The next cell says what is missing.
if len(df_valid):
    _longest = int(df_valid['n_words'].idxmax())
    _same_type = [j for j, ix in enumerate(pick_idx)
                  if df_valid.at[ix, 'item_type'] == df_valid.at[_longest, 'item_type']]
    if _longest not in pick_idx and _same_type:
        pick_idx[_same_type[-1]] = _longest

smoke_df = df_valid.loc[pick_idx].reset_index(drop=True)
print(f'Smoke draw: {len(smoke_df)} responses.')

Smoke draw: 20 responses.


In [29]:
# === CHECK THE DRAW BEFORE SPENDING ANYTHING ON IT ===
# WHAT THIS CELL DOES: confirms the rows drawn are the rows the plan above says,
# that all three item types are among them, and that twenty rows means twenty
# different passages.
#
# WHY IT IS A CHECK AND NOT A SENTENCE. "All three item types must be exercised"
# is true of the PLAN; whether it is true of the DRAW depends on the corpus. If
# an export column failed to coalesce in notebook 00, or if every answer to one
# item fell below the word floor, that item quietly leaves the draw: twenty calls
# go out on two item types, the third item's prompt is never sent to any model,
# and the smoke test still reports twenty out of twenty and pronounces the paid
# run safe. E3 leans hardest on the ranking item, and the ranking item is the one
# most likely to go missing (Q333 in the prereg, Priorit_Explain in the export).

# Counted over PER_ITEM rather than over what the draw happens to contain, so an
# item type with no rows at all reads as 0 instead of vanishing from the table.
# An item that disappears from its own coverage report is the failure this cell
# exists to catch, and it must not be able to hide in the shape of the report.
DRAWN = {it: int((smoke_df['item_type'] == it).sum()) for it in PER_ITEM}

# The tripwire for the swap in the cell above: forcing the longest row in by
# overwriting a slot of a different item type shows up here as five ranking rows
# where the plan said six.
assert DRAWN == EXPECTED_DRAW, (
    f'The draw does not match the plan: drew {DRAWN}, expected {EXPECTED_DRAW}. '
    'The row counts per item type are what make each of the three item prompts '
    'get tested.')

# One row per drawn passage. The pools are sampled without replacement and the
# corpus was checked for duplicate keys in the Load cell, so this holds; it is
# checked rather than assumed because every number printed from here on is a
# count of rows. A passage drawn twice would send one participant's answer to a
# leg twice and report twenty passages' worth of coverage over nineteen.
assert not smoke_df.duplicated(['response_id', 'item_type']).any(), \
    'the same (response_id, item_type) was drawn into the smoke sample twice'

# An item type that could not be drawn AT ALL is a corpus problem rather than a
# code problem, so it is reported rather than raised: the codebook below is still
# worth writing, and notebook 06 cannot run without it. What it must not do is
# pass, so it turns the verdict at the end of this notebook red.
ITEMS_MISSING = [it for it, n in EXPECTED_DRAW.items() if n == 0]
DRAW_OK = not ITEMS_MISSING
if ITEMS_MISSING:
    print('*** NO SCORABLE ROWS AT ALL FOR: ' + ', '.join(ITEMS_MISSING) + ' ***')
    print('  That item prompt will not be sent to any model, so this smoke test')
    print('  cannot prove it. A whole item missing from openends_long.csv means an')
    print('  export column did not coalesce in notebook 00. Check the per-item')
    print('  counts printed in the Load cell above before reading anything else.')

print(f'Drawn: {DRAWN}.')

Drawn: {'backward': 7, 'forward': 7, 'ranking': 6}.


In [30]:
# === THE TWENTY PROMPTS, AND WHAT THEY COST ===
# One prompt per row, built by the one function, with no model argument anywhere
# in the path. The hash is what turns "every leg is sent the same text" from a
# claim into a printed number: the same twenty hashes go to all three legs, and
# a hash appearing twice means two drawn participants wrote the same answer word
# for word. That last case is honest and harmless, so it is reported rather than
# asserted -- but it does mean the draw tests fewer passages than it looks like.
smoke_prompt_sha = [hashlib.sha256(build_user_prompt(r.text, r.item_type).encode('utf-8')
                                   ).hexdigest()[:16] for r in smoke_df.itertuples()]
_distinct = len(set(smoke_prompt_sha))
if _distinct != len(smoke_df):
    print(f'  Note: {len(smoke_df)} rows produced only {_distinct} distinct prompts, '
          'so two drawn participants wrote the same answer word for word.')

print(f'{len(smoke_df)} prompts built, {_distinct} of them distinct.')
if len(smoke_df):
    print(f'  word counts: shortest {int(smoke_df["n_words"].min())}, '
          f'longest {int(smoke_df["n_words"].max())}, '
          f'median {smoke_df["n_words"].median():.0f}')
    print(f'  the {TEXT_TRUNCATION_CHARS}-character window holds roughly '
          f'{TEXT_TRUNCATION_CHARS // 6} words, so the longest of these is the row '
          f'where\n  it bites.')
else:
    print('  Nothing was drawn, so no prompt was built and nothing can be sent.')
print(f'\nAbout to send {len(smoke_df) * len(MODELS)} joint calls '
      f'({len(MODELS)} legs x {len(smoke_df)} rows) plus {4 * len(MODELS)} split calls.')

20 prompts built, 20 of them distinct.
  word counts: shortest 4, longest 694, median 18
  the 1500-character window holds roughly 250 words, so the longest of these is the row where
  it bites.

About to send 60 joint calls (3 legs x 20 rows) plus 12 split calls.


In [31]:
# === PASS 1: THE JOINT NINE-FACET CALL ON EVERY CONFIGURED LEG ===
# WHY THIS RUNS ONE CALL AT A TIME. The paid run uses eight worker threads,
# because it makes thousands of calls and the machine spends its life waiting on
# the network. Here there are sixty calls and a different priority: a worker
# thread swallows the traceback and hands back a bare failure, and a bare failure
# is exactly what this notebook exists to explain. Sixty sequential calls take a
# minute or two and every error arrives with its own message attached.
SMOKE_COLS = (['pass_kind', 'model', 'model_id', 'response_id', 'item_type', 'n_words',
               'facet', 'parsed_ok', 'solo_value'] + FACETS
              + ['holistic', 'n_facets_returned', 'off_scale', 'prompt_sha16',
                 'raw_reply', 'text'])
smoke_rows = []

# === DID THE CELLS ABOVE ACTUALLY RUN IN THIS KERNEL? ===
# A restarted Colab session keeps the notebook and throws away every variable in
# it. Run on from the middle after that and Python reports the first missing
# name, which tells you nothing about why it is missing. This turns that into a
# sentence.
#
# THE COMMON CAUSE IS A PINNED INSTALL ON A WARM SESSION. Re-running the install
# cell after the kernel has already imported a package makes pip DOWNGRADE
# something in use, and Colab offers RESTART SESSION. Accepting is correct.
# Running on from where you were is what breaks.
_missing = [_n for _n in ('MODELS', 'smoke_df', 'score_one', 'CALL', 'FACETS',
           'RUBRIC', 'SMOKE_COLS') if _n not in globals()]
if _missing:
    raise NameError(
        'The cells above have not run in this kernel, so what these paid calls need is missing: '
        + ', '.join(_missing)
        + '.\n  Nothing is wrong with the notebook. This is what a restarted Colab'
          '\n  session looks like.'
          '\n  Fix: Runtime -> Run all. Always run this notebook from the top.')


for m in MODELS:
    print(f'\nLeg "{m}" ({MODELS[m]}): {len(smoke_df)} joint calls...')
    n_ok = 0
    for i, r in enumerate(smoke_df.itertuples()):
        raw, rec = score_one(r.text, r.item_type, m, return_raw=True)
        row = {c: np.nan for c in SMOKE_COLS}
        row.update({'pass_kind': 'joint', 'model': m, 'model_id': MODELS[m],
                    'response_id': r.response_id, 'item_type': r.item_type,
                    'n_words': r.n_words, 'facet': None,
                    'parsed_ok': rec is not None, 'solo_value': np.nan,
                    'off_scale': '', 'prompt_sha16': smoke_prompt_sha[i],
                    'raw_reply': raw, 'text': r.text})
        if rec is not None:
            n_ok += 1
            for f in FACETS:
                row[f] = rec[f]
            row['holistic'] = rec['holistic']
            # The values this reply sent that were not on the 0.0-1.0 scale,
            # written into the file beside the row that produced them, so the
            # spreadsheet shows which passage and which leg they came from.
            row['off_scale'] = '; '.join(rec['off_scale'])
            # How many of the nine the model actually returned. A reply that
            # parsed but came back with six facets is not a success: the missing
            # three are unmeasured, and in the paid run they would thin that
            # facet's rater count without thinning anything visible.
            row['n_facets_returned'] = int(sum(pd.notna(rec[f]) for f in FACETS))
        else:
            row['holistic'] = None
            row['n_facets_returned'] = 0
        smoke_rows.append(row)
        print('.' if rec is not None else 'X', end='')
    print(f'\n  {m}: {n_ok}/{len(smoke_df)} parsed.')
    # WHY, NOT JUST HOW MANY. A run of X's and a bare count is what this
    # notebook reported the first time it was used in anger, and it sent the
    # reader looking for a parser fault when the actual cause was a model
    # string that did not exist -- a sentence the API had supplied and the
    # notebook had thrown away. The first failing reply on a leg is printed
    # here so the leg explains itself on screen. Everything is still written
    # to the CSV; this only surfaces the one line that answers the question.
    if n_ok < len(smoke_df):
        _first = next(r for r in smoke_rows
                      if r['model'] == m and not r['parsed_ok'])
        print(f'  first failure on this leg, verbatim:')
        print(f'    {str(_first["raw_reply"])[:400]}')

if not MODELS:
    print('No leg is configured, so no calls were made. The rubric is still '
          'written below and the smoke result is reported as NOT RUN.')


Leg "claude" (claude-opus-4-6): 20 joint calls...
....................
  claude: 20/20 parsed.

Leg "gpt" (gpt-5.6-terra): 20 joint calls...
....................
  gpt: 20/20 parsed.

Leg "open" (meta-llama/Llama-3.3-70B-Instruct-Turbo): 20 joint calls...
....................
  open: 20/20 parsed.


In [32]:
# === PASS 2: THE SPLIT-CALL PROBE, SO THE E4 PROMPT IS PROVEN TOO ===
# WHY IT IS HERE. The paid run makes four EXTRA calls per response per model --
# desert and revenge, each with and without the co-occurrence sentence -- and
# those calls use a different prompt and a different one-key reply shape from the
# joint call. A joint-call smoke test that passed would say nothing at all about
# them, and the split-call estimate is the one that carries the primary weight
# for E4. Four calls per leg here proves the other prompt for well under a cent.
#
# One passage is enough. This is checking that the prompt is well formed and the
# reply parses, not what the scores are.
SOLO_PROBE_ROWS = 1

for m in MODELS:
    for i in range(min(SOLO_PROBE_ROWS, len(smoke_df))):
        r = smoke_df.iloc[i]
        for facet in ('desert', 'revenge'):
            for mixed in (True, False):
                raw, val = score_facet_solo(r['text'], r['item_type'], facet, m,
                                            mixed_anchor=mixed, return_raw=True)
                kind = 'solo' if mixed else 'solo_noanchor'
                row = {c: np.nan for c in SMOKE_COLS}
                row.update({'pass_kind': kind, 'model': m, 'model_id': MODELS[m],
                            'response_id': r['response_id'], 'item_type': r['item_type'],
                            'n_words': r['n_words'], 'facet': facet,
                            'parsed_ok': val is not None, 'solo_value': val,
                            'holistic': None, 'n_facets_returned': np.nan,
                            # A solo value off the 0.0-1.0 scale comes back as
                            # None, so it lands here as a failure; the value the
                            # model actually sent is in raw_reply on this row and
                            # in the off-scale report below.
                            'off_scale': '',
                            'prompt_sha16': hashlib.sha256(
                                build_solo_prompt(r['text'], r['item_type'], facet,
                                                  mixed_anchor=mixed).encode('utf-8')
                            ).hexdigest()[:16],
                            'raw_reply': raw, 'text': r['text']})
                smoke_rows.append(row)
                print(f'  {m:8s} {facet:8s} {kind:14s} -> {val}')

smoke = pd.DataFrame(smoke_rows, columns=SMOKE_COLS)
print(f'\nSmoke frame: {len(smoke)} records '
      f'({int((smoke["pass_kind"] == "joint").sum())} joint, '
      f'{int((smoke["pass_kind"] != "joint").sum())} split).')

  claude   desert   solo           -> 0.65
  claude   desert   solo_noanchor  -> 0.5
  claude   revenge  solo           -> 0.1
  claude   revenge  solo_noanchor  -> 0.1
  gpt      desert   solo           -> 0.35
  gpt      desert   solo_noanchor  -> 0.0
  gpt      revenge  solo           -> 0.0
  gpt      revenge  solo_noanchor  -> 0.0
  open     desert   solo           -> 0.8
  open     desert   solo_noanchor  -> 0.8
  open     revenge  solo           -> 0.0
  open     revenge  solo_noanchor  -> 0.0

Smoke frame: 72 records (60 joint, 12 split).


In [33]:
# === WHAT THE MODEL SAID, BESIDE WHAT THE PARSER MADE OF IT ===
# This is the cell the whole notebook is for. Everything else can be inferred
# from a count; this cannot. A parse rate of 20/20 with every facet coming back
# 0.5 is a passing number and a broken measurement, and the only way to see that
# is to read a reply. The accounting below catches a leg answering on the wrong
# SCALE, because 7 is not a number between 0 and 1; nothing catches a leg that
# returns a plausible 0.5 for everything except a human reading three replies.
#
# HOW TO READ A REPLY, NOW THAT NO LEG PREFILLS. Every leg should return a
# complete JSON object, so every raw reply below should START with an opening
# brace. That is the opposite of what this notebook expected before: the
# Anthropic leg used to have its assistant turn seeded with '{' and its text
# therefore began after it. A reply that does NOT start with '{' now means
# preamble leaked in past the system prompt. parse_json_safe recovers that at
# step 4, so it may still be counted as parsed -- but it is worth seeing,
# because a leg that leaks preamble on the smoke test will leak it on the
# corpus, and the recovery is a slice rather than a guarantee.
def show_one(row, why):
    print('=' * 78)
    print(f'{why}  |  leg: {row["model"]} ({row["model_id"]})')
    print(f'  row       : {row["response_id"]} [{row["item_type"]}], '
          f'{row["n_words"]} words, pass: {row["pass_kind"]}')
    print(f'  passage   : {str(row["text"])[:320]}')
    print('  --- RAW REPLY, exactly as the model sent it ---')
    print('  ' + str(row['raw_reply'])[:1200].replace('\n', '\n  '))
    print('  --- PARSED, exactly what came out of the parser ---')
    if not row['parsed_ok']:
        print('  NOTHING. This reply did not parse. In the paid run it would be')
        print('  kept as an explicit unscored row, never dropped and never zeroed.')
    elif row['pass_kind'] == 'joint':
        for f in FACETS:
            print(f'    {f:16s} {row[f]}')
        print(f'    holistic         {row["holistic"]}')
        print(f'    facets returned  {int(row["n_facets_returned"])}/9')
    else:
        print(f'    {row["facet"]} = {row["solo_value"]}')

if len(smoke):
    for m in MODELS:
        leg = smoke[smoke['model'] == m]
        for _, row in leg.head(SHOW_RAW_N).iterrows():
            show_one(row, 'SAMPLE REPLY')
        # Every failure, regardless of SHOW_RAW_N. A failure you cannot read is a
        # failure you cannot fix. Capped at six per leg only because a leg that
        # fails uniformly has ONE cause, not sixty, and sixty copies of it would
        # bury the legs that worked.
        fails = leg[~leg['parsed_ok'].astype(bool)]
        for _, row in fails.head(6).iterrows():
            show_one(row, '*** PARSE FAILURE ***')
        if len(fails) > 6:
            print(f'  ... and {len(fails) - 6} more failures on leg {m}. A leg that '
                  f'fails this uniformly\n      has one cause: check the model string '
                  f'and the key before reading further.')
    print('=' * 78)
else:
    print('Nothing to show: no leg was configured, so no reply was received.')

SAMPLE REPLY  |  leg: claude (claude-opus-4-6)
  row       : R_37pERk7YMXWU3ct [backward], 5 words, pass: joint
  passage   : because he still killed someone
  --- RAW REPLY, exactly as the model sent it ---
  {"desert":0.8,"proportionality":0.0,"censure":0.0,"moral_balance":0.0,"revenge":0.0,"suffering":0.0,"deterrence":0.0,"incapacitation":0.0,"rehabilitation":0.0,"holistic":"principled","evidence":{"desert":"he still killed someone"}}
  --- PARSED, exactly what came out of the parser ---
    desert           0.8
    proportionality  0.0
    censure          0.0
    moral_balance    0.0
    revenge          0.0
    suffering        0.0
    deterrence       0.0
    incapacitation   0.0
    rehabilitation   0.0
    holistic         principled
    facets returned  9/9
SAMPLE REPLY  |  leg: claude (claude-opus-4-6)
  row       : R_3mVNAWyX2u7zItY [backward], 25 words, pass: joint
  passage   : Darryl still needs to answer to the crime he committed, but he is no longer a threat so I will 

In [34]:
# === PARSE-RATE ACCOUNTING: THE GO / NO-GO READ, PART 1 (THE JOINT CALL) ===
# WHY PARSE FAILURES ARE KEPT AS EXPLICIT UNSCORED ROWS RATHER THAN DROPPED.
# A dropped failure is invisible, and invisible loss is not random. The replies
# most likely to fail are the long, graphic, vengeful ones -- the same responses
# that carry the revenge and suffering signal -- so silently discarding them
# would shrink exactly the tail the study is trying to measure while leaving
# every printed count looking healthy. Keeping the row means:
#   coverage stays honest      the denominator is always the number of responses
#                              attempted, so the parse rate is a reported number
#                              rather than an inferred one
#   the mean skips, not zeros  an unscored row contributes nothing to the
#                              ensemble mean; a dropped row would have made the
#                              denominator lie, and a 0.0 would have made the
#                              facet look absent
#   failure is analysable      per-model, per-item non-response can be counted
#                              and checked for correlation with content
#   the archive is complete    the OSF record shows what was attempted, not only
#                              what succeeded
#
# WHAT THE NUMBERS BELOW MEAN. Expect a parse rate within a point or two of 100%
# on every leg, nine facets returned on average, and no off-scale values at all.
# A leg sitting far below the others is not a noisier rater, it is a broken one
# -- wrong prefill, wrong model string, JSON mode unsupported, a refusal pattern
# -- and FAILURE_LOG says which. Reading a low-coverage leg's agreement as
# disagreement would be wrong.
LEG_JOINT_OK = {}
print('Per leg, joint calls:')
for m in MODELS:
    leg = smoke[(smoke['pass_kind'] == 'joint') & (smoke['model'] == m)]
    ok = int(leg['parsed_ok'].astype(bool).sum())
    mean_facets = leg.loc[leg['parsed_ok'].astype(bool), 'n_facets_returned'].mean()
    off = int((leg['off_scale'].fillna('').astype(str).str.len() > 0).sum())
    # Three conditions, not one. Every reply parsed, every reply came back with
    # all nine facets, and no value arrived off the scale. A leg can pass the
    # first two and fail the third, and that is the case worth catching here.
    clean = (ok == len(leg) and pd.notna(mean_facets)
             and mean_facets >= len(FACETS) and off == 0)
    LEG_JOINT_OK[m] = bool(clean)
    # A leg that parsed nothing has no average to report. It prints as "--",
    # which says the quantity could not be computed; a printed "nan" reads like a
    # value, and a blank reads like a zero.
    shown = '--' if pd.isna(mean_facets) else f'{mean_facets:.1f}'
    print(f'  {m:8s} parsed {ok}/{len(leg)}   '
          f'mean facets returned {shown}/{len(FACETS)}   '
          f'replies with off-scale values {off}'
          f'{"" if clean else "   <-- LOOK AT THIS LEG"}')
# A leg with no key is NOT RUN. It is named on its own line, never left as a
# blank row that a reader could mistake for a leg that ran and failed.
for k in NOT_RUN:
    print(f'  {k:8s} NOT RUN -- no API key is configured for this leg.')

Per leg, joint calls:
  claude   parsed 20/20   mean facets returned 9.0/9   replies with off-scale values 0
  gpt      parsed 20/20   mean facets returned 9.0/9   replies with off-scale values 0
  open     parsed 20/20   mean facets returned 9.0/9   replies with off-scale values 0


In [35]:
# === THE GO / NO-GO READ, PART 2: THE SPLIT-CALL ARM AND THE VERDICT ===
# WHY THE SPLIT CALLS GET THEIR OWN PER-LEG LINE, AND WHY A LEG IS NOT PROVEN
# WITHOUT THEM. They use a different prompt and a different one-key reply shape,
# so they can fail on a leg whose every joint call passed: a provider's JSON mode
# that insists on more than one key, a model that answers a single-facet question
# in a sentence. The split-call estimate is the one that carries the primary
# weight for E4 (design section 0), so a leg with twenty clean joint calls and
# four failed split calls is NOT a proven leg -- and recording it as one would
# put a false assurance into the file notebook 06 reads.
LEGS_CLEAN = []
split = smoke[smoke['pass_kind'] != 'joint']
print('\nPer leg, split calls (desert and revenge, each with and without the '
      'co-occurrence sentence):')
for m in MODELS:
    leg = split[split['model'] == m]
    ok = int(leg['parsed_ok'].astype(bool).sum())
    clean = LEG_JOINT_OK.get(m, False) and len(leg) > 0 and ok == len(leg)
    why = ''
    if not clean:
        why = ('   <-- NOT PROVEN: '
               + ('its joint calls' if not LEG_JOINT_OK.get(m, False)
                  else 'these split calls')
               + ' did not come back clean')
    print(f'  {m:8s} parsed {ok}/{len(leg)}{why}')
    if clean:
        LEGS_CLEAN.append(m)
if MODELS and not len(split):
    print('  none came back at all. The E4 prompt is UNPROVEN.')

# The off-scale report. Printed with the values themselves, because "3 off-scale
# values" tells a reader nothing and "desert=7, revenge=7" tells them the leg is
# answering on a seven-point scale.
if OFF_SCALE:
    print(f'\n*** {len(OFF_SCALE)} FACET VALUES CAME BACK OUTSIDE {FACET_MIN}-{FACET_MAX} ***')
    for _e in OFF_SCALE[:12]:
        print(f'    leg {_e["model"]:8s} {_e["where"]:14s} {_e["value"]}')
    if len(OFF_SCALE) > 12:
        print(f'    ... and {len(OFF_SCALE) - 12} more')
    print('  Those values are recorded as UNMEASURED, never clipped into range.')
    print('  A leg answering 7 out of 10 where the codebook asks for 0.0 to 1.0 is')
    print('  reading the scale wrong, and no parse rate would ever show it.')

# THE ONE FLAG THE FINAL VERDICT READS, AND THE ONE WRITTEN INTO rubric.json AS
# all_clear. Every part of it has to be true: calls were made at all; the draw
# covered all three item types; nothing came back off the scale; every configured
# leg was clean on BOTH passes; and no leg is missing. The last condition is
# there because prereg 6.7 requires at least three developers, so a panel with a
# leg unconfigured is not a panel the design accepts, however clean the other two
# look -- and a file recording all_clear on a two-leg run would say otherwise.
SMOKE_OK = (bool(len(smoke)) and DRAW_OK and not OFF_SCALE
            and set(LEGS_CLEAN) == set(MODELS) and not NOT_RUN)
print(f'\nLegs proven on both passes: ' + (', '.join(LEGS_CLEAN) if LEGS_CLEAN else 'none'))


Per leg, split calls (desert and revenge, each with and without the co-occurrence sentence):
  claude   parsed 4/4
  gpt      parsed 4/4
  open     parsed 4/4

Legs proven on both passes: claude, gpt, open


In [36]:
# === THE VERBATIM FAILURES ===
# The raw text of the first few unparseable replies. Prose preamble, a refusal, a
# markdown fence, a truncation and an authentication error all arrive as the same
# failure and call for completely different fixes; this is the thing that tells
# them apart, and it is the first thing to read when a leg's parse rate comes
# back low.
n_calls  = int(len(smoke))
n_failed = int((~smoke['parsed_ok'].astype(bool)).sum()) if n_calls else 0
if not n_calls:
    # An empty log after zero calls is not an all-clear, and the line that would
    # otherwise print here reads exactly like one.
    print('\nFAILURE_LOG is empty because NOTHING WAS TESTED: not one call was made.')
    if not MODELS:
        print('  No leg is configured, so there was nothing to send a prompt to.')
    else:
        print('  The legs are configured, but the draw was empty, so there was')
        print('  nothing to send them.')
    print('  This is not a clean smoke test; it is an absent one.')
elif FAILURE_LOG:
    print(f'\nFAILURE_LOG ({len(FAILURE_LOG)} kept, capped at 5):')
    for entry in FAILURE_LOG:
        print(f'  leg {entry["model"]} | prompt starts: {entry["prompt_first120"][:80]}...')
        print(f'    raw: {entry["raw_response"][:300]}')
elif n_failed:
    # An empty log next to a pile of failures is itself informative, and saying
    # so beats a line that reads as an all-clear.
    print(f'\nFAILURE_LOG is empty even though {n_failed} calls failed. That means the')
    print('failures were errors rather than unparseable replies: no reply text came')
    print('back at all, so there was nothing to keep. The error text is on the raw')
    print('line of each failed row printed above.')
else:
    print(f'\nFAILURE_LOG is empty: all {n_calls} replies that came back parsed.')


FAILURE_LOG is empty: all 72 replies that came back parsed.


## Save and check

In [37]:
# === WRITE THE RUBRIC ===
# rubric.json is the single copy of the codebook in the whole pipeline. The paid
# run reads its prompts from this file rather than holding its own transcription
# of them, which is what makes "the rubric is identical across models" a property
# of the code and not a thing anyone has to remember.
#
# The smoke result is recorded inside it, so the file says not only what the
# codebook is but which model strings it was actually proven against and on what
# date. A rubric that has never been fired at a live model is a document; one
# with this block filled in is a tested instrument.
def _rate(m, joint=True):
    """Reported as 'parsed/attempted', never as a bare percentage: the
    denominator is the number of calls sent, so a leg that failed everything
    reads as 0/20 rather than as a blank that could be mistaken for a pass."""
    sel = smoke['pass_kind'] == 'joint' if joint else smoke['pass_kind'] != 'joint'
    leg = smoke[sel & (smoke['model'] == m)]
    return f'{int(leg["parsed_ok"].astype(bool).sum())}/{len(leg)}'

RUBRIC['smoke_test'] = {
    'run_at':           datetime.datetime.now().isoformat(timespec='seconds'),
    'n_rows':           int(len(smoke_df)),
    # WHICH CLIENT LIBRARIES PRODUCED THESE SCORES. The model id says which
    # model was asked; this says which library did the asking, and after
    # anthropic 1.0.0 removed temperature that is not a detail. Recorded
    # here so the fact travels with the data rather than living only in a
    # Colab output cell that is gone when the runtime is.
    'client_versions':  CLIENT_VERSIONS,
    # Which item prompts were actually exercised. A missing item here means the
    # prompt for it was never sent, whatever the parse rates say.
    'item_types_tested': DRAWN,
    'legs_run':         {m: MODELS[m] for m in MODELS},
    # The legs the rubric is actually PROVEN against: every joint call parsed
    # with all nine facets on the 0.0-1.0 scale AND every split call parsed. A
    # leg that ran and failed either pass is in legs_run and not here, which is a
    # different fact from a leg that never ran at all.
    'legs_proven':      LEGS_CLEAN,
    'not_run':          NOT_RUN,      # named, never a blank that reads as a failure
    'joint_parse_rate': {m: _rate(m, True) for m in MODELS},
    'split_parse_rate': {m: _rate(m, False) for m in MODELS},
    'off_scale_values': len(OFF_SCALE),
    'all_clear':        bool(SMOKE_OK),
}

with open('rubric.json', 'w', encoding='utf-8') as fh:
    json.dump(RUBRIC, fh, indent=1, ensure_ascii=False)
done('rubric.json', rows=len(FACETS),
     note=f'one written codebook, {len(FACETS)} facets, fingerprint {RUBRIC_SHA16}; '
          f'numeric anchor targets held out, evidence spans optional; the codebook '
          f'does not depend on RUN_MODE and is byte-identical in either mode')

  [wrote] rubric.json  (9 rows, 6,015 bytes)


In [38]:
# === WRITE THE SMOKE RESULTS ===
# Every call is in here, successes and failures alike, with the raw reply text
# beside the parsed values. It is small enough to open in a spreadsheet and read
# line by line, which is the point: the parse rate says whether the replies were
# well formed, and only the text says whether they were sensible.
#
# THE KEY OF THIS FILE is (pass_kind, model, response_id, item_type, facet): one
# row per CALL. It is deliberately not one row per participant per item, because
# the same passage is called once on each leg and four more times for the split
# probe. What must hold is that no single call is written twice, and that is
# checked here rather than assumed, because every parse rate above is a count of
# these rows and a duplicated call would inflate both halves of the fraction.
_CALL_KEY = ['pass_kind', 'model', 'response_id', 'item_type', 'facet']
_dupe_calls = int(smoke.duplicated(_CALL_KEY).sum())
assert _dupe_calls == 0, (
    f'{_dupe_calls} calls appear twice in the smoke frame under the same '
    f'{tuple(_CALL_KEY)}. Every parse rate printed above counts these rows.')

smoke.to_csv('smoke_test_results.csv', index=False)
done('smoke_test_results.csv', rows=len(smoke),
     note=f'{len(MODELS)} legs x {len(smoke_df)} rows joint, plus the split-call probe; '
          f'one row per call, keyed on {"/".join(_CALL_KEY)}; '
          f'rubric fingerprint {RUBRIC_SHA16}')

  [wrote] smoke_test_results.csv  (72 rows, 49,919 bytes)


In [39]:
# === WHAT THIS NOTEBOOK PRODUCED, AND WHAT TO RUN NEXT ===
print(f'NOTEBOOK {NB_ID} -- {NB_NAME}: done.\n')
print(f'  rubric.json               the one codebook, fingerprint {RUBRIC_SHA16}')
print(f'  smoke_test_results.csv    {len(smoke)} calls, raw reply beside parsed record')
print('\n  legs proven : ' + (', '.join(LEGS_CLEAN) if LEGS_CLEAN else 'none')
      + '   (clean on the joint pass AND the split pass)')
print('  legs run    : ' + (', '.join(MODELS) if MODELS else 'none'))
print('  legs not run: ' + (', '.join(NOT_RUN) if NOT_RUN else 'none'))
print(f'  items tested: ' + ', '.join(f'{k} {v}' for k, v in DRAWN.items()))

print('\nRead this before spending anything:')
# SMOKE_OK already carries all of it: calls were made, all three item types were
# drawn, every leg was clean on both passes, nothing came back off the scale, and
# no leg is sitting unconfigured.
if SMOKE_OK:
    print('  Every configured leg parsed every joint reply and every split reply,')
    print('  returned all nine facets, and returned every value on the 0.0-1.0')
    print('  scale. The rubric and the parser are working; the paid run is safe.')
else:
    print('  The smoke test did NOT come back clean. Do not start the paid run yet.')
    print('  Work through, in this order:')
    print('    1. any leg marked NOT RUN            -> its API key is not configured')
    print('    2. any placeholder model string      -> pin the dated version string')
    print('    3. any item type missing from the draw -> an open-end is missing from')
    print('       openends_long.csv; re-run notebook 00 before scoring two items')
    print('    4. any off-scale value               -> the leg is reading "0.0 to 1.0"')
    print('       as some other scale; nothing downstream would catch that')
    print('    5. any parse failure printed above   -> read the raw reply; a refusal,')
    print('       a prose preamble and a truncation each need a different fix')
    print('    6. split calls failing on a leg whose joint calls passed -> the E4')
    print('       prompt, not the codebook: read the raw reply on a solo row')
    print('    7. fewer than nine facets returned   -> the reply is being cut off;')
    print('       raise max_tokens before the missing facets become quiet NaNs')
print('\n  And read three or four of the sample replies above even if the counts are')
print('  clean. A leg can parse perfectly, answer on the right scale, and still be')
print('  scoring the vignette instead of the writer, and no count will show you that.')

print('\nNext: the paid ensemble pass (notebook 06). It reads rubric.json, so the')
print('codebook it scores under is the one proven here and not a second copy of it.')

NOTEBOOK 05 -- rubric_and_smoke_test: done.

  rubric.json               the one codebook, fingerprint fef0381e46892391
  smoke_test_results.csv    72 calls, raw reply beside parsed record

  legs proven : claude, gpt, open   (clean on the joint pass AND the split pass)
  legs run    : claude, gpt, open
  legs not run: none
  items tested: backward 7, forward 7, ranking 6

Read this before spending anything:
  Every configured leg parsed every joint reply and every split reply,
  returned all nine facets, and returned every value on the 0.0-1.0
  scale. The rubric and the parser are working; the paid run is safe.

  And read three or four of the sample replies above even if the counts are
  clean. A leg can parse perfectly, answer on the right scale, and still be
  scoring the vignette instead of the writer, and no count will show you that.

Next: the paid ensemble pass (notebook 06). It reads rubric.json, so the
codebook it scores under is the one proven here and not a second copy of 